In [ ]:
cd(joinpath(@__DIR__, ".."))   # run from the repository root

# Higher-order Whitney forms & DEC on a refined mesh

This notebook constructs and validates **higher-order finite-element exterior calculus (FEEC / DEC)** operators on a doubly-periodic mesh, where the **Delaunay triangles are the primal complex**.

**Road map**
1. **Whitney basics** — load a mesh, the lowest-order Whitney 1-forms and their defining property.
2. **Higher-order Whitney basis** $\lambda^\alpha W_e$ spanning $P_2^-\Lambda^1$.
3. **Coarse→fine medial refinement** and the per-triangle stencils (fine cells, primal edges, dual edges).
4. **Reconstruction operators** $I_h^1$ (1-forms), $R_0$ ($P_2\Lambda^0$), $R_2$ ($P_1\Lambda^2$) by least squares, plus convergence sweeps.
5. **Commutation** $d\circ I_h^k = I_h^{k+1}\circ\bar d_k$ (the FEEC property), tested to machine precision and for convergence.
6. **Higher-order Whitney Hodge star** $\star_1$ mapping fine primal edges to fine dual edges, with a periodic convergence test.

**This cell** loads the packages, applies a `Base.similar` workaround for a TensorsLite/Makie dispatch clash, and `include`s the project sources `juliaDEC.jl` and `mesh_refinement_map.jl`.

In [ ]:
using VoronoiMeshes,TensorsLite, TensorsLiteGeometry,  SmallCollections, LinearAlgebra, NCDatasets
using StaticArrays, SparseArrays, QuadGK
using CairoMakie, LaTeXStrings
using LowRankApprox

# Workaround for TensorsLite/Makie clash: TensorsLite pirates
# `similar(::Broadcasted, ::Type{T<:AbstractTensor})`, which also catches
# `T === Union{}` because `Union{} <: AbstractTensor`. Makie hits this path
# whenever it constructs an empty `Attributes()` (every `Figure()`).
Base.similar(bc::Base.Broadcast.Broadcasted, ::Type{Union{}}) =
    Array{Union{}}(undef, length.(axes(bc)))

include("src/juliaDEC.jl");
include("src/mesh_refinement_map.jl");

### Load and plot a mesh

`VoronoiMesh("….nc")` loads a doubly-periodic Delaunay/Voronoi mesh from NetCDF; `plotmesh` draws the primal complex. Throughout, the **Delaunay triangles are the primal complex**.

In [ ]:
mesh = VoronoiMesh("meshes/mesh2_regular.nc")
plotmesh(mesh)

### Triangle connectivity — `get_triangle`

Returns a triangle's vertex positions, its global edge indices, and the edge orientations — the data every per-triangle routine below relies on.

In [ ]:
points, edge_indices, edge_orientations = get_triangle(mesh, 7)


### Lowest-order Whitney basis on a reference triangle

Plots the three lowest-order Whitney 1-forms on the unit reference triangle using `whitney_reconstruction_on_x_periodic`, which returns the $3\times2$ matrix of basis vectors at a point.

In [ ]:
basis_number = 3
p1 = [0.0, 0.0]
p2 = [1.0, 0.0]
p3 = [0.0, 1.0]

xs = range(-0.2, 1.2, length=12)
ys = range(-0.2, 1.2, length=12)
points = [ [x, y] for x in xs, y in ys ]
fig = Figure(size = (700, 500))

for k=1:basis_number

    us = zeros(length(points))
    vs = zeros(length(points))
    x_plot = zeros(length(points))
    y_plot = zeros(length(points))

    # Compute vector field values
    for (i, p) in enumerate(points)  
        W = whitney_reconstruction_on_x_periodic(p, p1, p2, p3)
        
        # Extract first row (first basis function)
        # W is a 3x2 matrix, so W[1, :] is the vector [u, v]
        us[i] = W[k, 1]
        vs[i] = W[k, 2]
        x_plot[i] = p[1]
        y_plot[i] = p[2]
    end

    # Plot
    ax = Axis(fig[1, k], aspect = DataAspect(), title = "Whitney Basis Function $k")

    # Plot the triangle for reference
    lines!(ax, [p1[1], p2[1], p3[1], p1[1]], [p1[2], p2[2], p3[2], p1[2]], color = :black, linewidth=2)

    # Plot the vector field
    arrows!(ax, x_plot, y_plot, us, vs, lengthscale = 0.1, color = :blue)
end
fig


### Whitney basis on a mesh triangle

Evaluates `whitney_basis_functions_on_triangle_on_x` on a grid and plots the first basis vector field over a real mesh triangle, with the dual mesh (`plotdualmesh!`) and circumcenter drawn for context.

In [ ]:
# Load the mesh
mesh = VoronoiMesh("meshes/mesh1_regular.nc")

triangle_idx = 2
points, edge_indices, edge_orientations = get_triangle(mesh, triangle_idx)
p1, p2, p3 = points
first_edge = edge_indices[1]
edge_position = mesh.edges.position[first_edge]
new_edge_position = closest(mesh.vertices.position[triangle_idx],edge_position, mesh.x_period, mesh.y_period)
xs = range(-0.2, 1.2, length=16)
ys = range(-0.2, 1.2, length=16)
grid_points = [ [x, y] for x in xs, y in ys ]
fig = Figure(size = (700, 500))

# Initialize arrays for plotting
us = zeros(length(grid_points))
vs = zeros(length(grid_points))
x_plot = zeros(length(grid_points))
y_plot = zeros(length(grid_points))

# Compute vector field values
for (i, p) in enumerate(grid_points)
    W, _, _ = whitney_basis_functions_on_triangle_on_x(mesh, p, triangle_idx)
    us[i] = W[1, 1]
    vs[i] = W[1, 2]
    x_plot[i] = p[1]
    y_plot[i] = p[2]
end

# Plot
fig = Figure(size=(600, 600))
ax = Axis(fig[1, 1], aspect = DataAspect(), title = "Whitney Basis for Triangle $triangle_idx (Edge $(edge_indices[1]))")


plotdualmesh!(mesh)
scatter!(ax, p1[1], p1[2], color = :red, markersize = 12)
scatter!(ax, p2[1], p2[2], color = :red, markersize = 12)
#plot the triangle
lines!(ax, [p1[1], p2[1], p3[1], p1[1]], [p1[2], p2[2], p3[2], p1[2]], color = :black, linewidth=3)
# Plot the vector field
arrows!(ax, x_plot, y_plot, us, vs, lengthscale = 0.05, color = :blue)
scatter!(mesh.vertices.position.x[triangle_idx], mesh.vertices.position.y[triangle_idx], color = :green, markersize = 22)
fig

### Verify the defining property of the Whitney basis

Checks the duality $\int_{e_j}W_i\cdot d\ell=\delta_{ij}$ by integrating each basis function over the three edges with `edge_integral` — the property that makes edge circulations the natural degrees of freedom for 1-forms.

In [ ]:
triangle_idx = 2
points, edge_indices, edge_orientations = get_triangle(mesh, triangle_idx)
p1, p2, p3 = points

println("Verifying Whitney basis properties for Triangle $triangle_idx")
println("Vertices: p1=$(p1), p2=$(p2), p3=$(p3)")
println("Edge indices: $(edge_indices)")
println("Edge orientations: $(edge_orientations)")

function whitney_basis_1(x)
    W = whitney_reconstruction_on_x_periodic(x, p1, p2, p3)
    return [W[1, 1], W[1, 2]]
end 
function whitney_basis_2(x)
    W = whitney_reconstruction_on_x_periodic(x, p1, p2, p3)
    return [W[2, 1], W[2, 2]]
end
function whitney_basis_3(x)
    W = whitney_reconstruction_on_x_periodic(x, p1, p2, p3)
    return [W[3, 1], W[3, 2]]
end
xp = mesh.x_period
yp = mesh.y_period

# Edge 1: p1 -> p2
int_edge1 = edge_integral(whitney_basis_1, p1, p2, xp, yp)

# Edge 2: p2 -> p3
int_edge2 = edge_integral(whitney_basis_1, p2, p3, xp, yp)

# Edge 3: p3 -> p1
int_edge3 = edge_integral(whitney_basis_1, p3, p1, xp, yp)

println("\nIntegration Results for First Whitney Basis Function (W1):")
println("Integral along Edge 1 (p1->p2): $int_edge1  (Expected: 1.0)")
println("Integral along Edge 2 (p2->p3): $int_edge2  (Expected: 0.0)")
println("Integral along Edge 3 (p3->p1): $int_edge3  (Expected: 0.0)")
# Edge 1: p1 -> p2
int_edge1 = edge_integral(whitney_basis_2, p1, p2, xp, yp)

# Edge 2: p2 -> p3
int_edge2 = edge_integral(whitney_basis_2, p2, p3, xp, yp)

# Edge 3: p3 -> p1
int_edge3 = edge_integral(whitney_basis_2, p3, p1, xp, yp)

println("\nIntegration Results for First Whitney Basis Function (W2):")
println("Integral along Edge 1 (p1->p2): $int_edge1  (Expected: 0.0)")
println("Integral along Edge 2 (p2->p3): $int_edge2  (Expected: 1.0)")
println("Integral along Edge 3 (p3->p1): $int_edge3  (Expected: 0.0)")

# Edge 1: p1 -> p2
int_edge1 = edge_integral(whitney_basis_3, p1, p2, xp, yp)

# Edge 2: p2 -> p3
int_edge2 = edge_integral(whitney_basis_3, p2, p3, xp, yp)

# Edge 3: p3 -> p1
int_edge3 = edge_integral(whitney_basis_3, p3, p1, xp, yp)

println("\nIntegration Results for First Whitney Basis Function (W3):")
println("Integral along Edge 1 (p1->p2): $int_edge1  (Expected: 0.0)")
println("Integral along Edge 2 (p2->p3): $int_edge2  (Expected: 0.0)")
println("Integral along Edge 3 (p3->p1): $int_edge3  (Expected: 1.0)")

### Discretize and interpolate a field

`discretize_vector_field_dual` turns a vector field into a primal 1-cochain (edge circulations); `whitney_interpolation_on_triangle` reconstructs it with the lowest-order Whitney basis on a triangle. The cell compares the reconstruction against the exact field at the circumcenter.

In [ ]:
idx = 2
v = x -> [-2,1]
v_mesh = discretize_vector_field_dual(mesh, v)
whitney_approximation = whitney_interpolation_on_triangle(mesh, idx, v_mesh)
circumcenter = mesh.vertices.position[idx] 
error_vector = v(circumcenter) - whitney_approximation

## Higher-order Whitney basis

The lowest-order Whitney 1-form $W_e$ (one per edge) is enriched to a **degree-$r$** space by multiplying it with barycentric monomials $\lambda^\alpha$, $|\alpha|=r-1$. This spans the trimmed polynomial space $P_r^-\Lambda^1$. The cells below define the multi-index machinery ($\lambda^\alpha$), evaluate the enriched basis $\lambda^\alpha W_e$ on a triangle, and visualize each component.

### Multi-indices and barycentric monomials

- `generate_multi_indices(r)` — all $\alpha=(\alpha_0,\alpha_1,\alpha_2)$ with $|\alpha|=r$, indexing the barycentric powers of the degree-$r$ space;
- `eval_monomial(\lambda,\alpha)` — evaluates $\lambda^\alpha=\lambda_0^{\alpha_0}\lambda_1^{\alpha_1}\lambda_2^{\alpha_2}$.

The next cell lists the degree-2 indices.

In [ ]:
#Define a MultiIndex type
const MultiIndex = NTuple{3, Int}

"""
    generate_multi_indices(r::Int)

Generates the set I(r, 2) of all multi-indices α = (α₀, α₁, α₂) such that Σα = r.
Returns a vector of MultiIndex.
"""
function generate_multi_indices(r::Int)
    indices = Vector{MultiIndex}()
    for i in 0:r
        for j in 0:(r-i)
            k = r - i - j
            push!(indices, (i, j, k))
        end
    end
    return indices
end

"""
    eval_monomial(lambda::Vector{T}, alpha::MultiIndex)

Evaluates the scalar monomial λ^α = λ₀^α₀ * λ₁^α₁ * λ₂^α₂
"""
function eval_monomial(lambda::Vector{T}, alpha::MultiIndex) where T
    return (lambda[1]^alpha[1]) * (lambda[2]^alpha[2]) * (lambda[3]^alpha[3])
end

In [ ]:
generate_multi_indices(2)


### Higher-order Whitney basis $\lambda^\alpha W_e$ — `compute_higher_order_whitney_basis`

Evaluates the enriched basis at a point $x$: wraps the triangle and $x$ to a common periodic tile, computes barycentric coordinates $\lambda$ and their gradients $d\lambda$, forms the three lowest-order Whitney forms $W_e=\lambda_a\,d\lambda_b-\lambda_b\,d\lambda_a$ (signed by the global edge orientation), then multiplies each by every monomial $\lambda^\alpha$. Returns a dict `(edge, α) → [vx, vy]`. The next cells evaluate and inspect it.

In [ ]:
"""
    compute_higher_order_whitney_basis(mesh, triangle_idx, x, poly_degree)

Computes the higher-order Whitney basis functions λ^α * W_edge at a point x.

Arguments:
- `mesh`: The VoronoiMesh
- `triangle_idx`: Index of the triangle
- `x`: Evaluation point [x, y]
- `poly_degree`: Degree of the polynomial part (multi-index sum)

Returns:
- Dictionary mapping (alpha, edge_local_index) -> Vector value [vx, vy]
"""
function compute_higher_order_whitney_basis(mesh::VoronoiMesh, triangle_idx::Int, x::Vec2Dxy{Float64}, poly_degree::Int=2)
    points, edge_indices, edge_orientations = get_triangle(mesh, triangle_idx)
    center_pos = TensorsLite.Vec(x=mesh.vertices.position.x[triangle_idx], 
                                 y=mesh.vertices.position.y[triangle_idx])
    xp, yp = mesh.x_period, mesh.y_period

    # Wrap triangle vertices into the same periodic tile as the center
    p1 = closest(center_pos, TensorsLite.Vec(x=points[1][1], y=points[1][2]), xp, yp)
    p2 = closest(center_pos, TensorsLite.Vec(x=points[2][1], y=points[2][2]), xp, yp)
    p3 = closest(center_pos, TensorsLite.Vec(x=points[3][1], y=points[3][2]), xp, yp)

    x_vec = TensorsLite.Vec(x=x[1], y=x[2])
    
    # Map x into the same periodic tile as the triangle center
    x_closest = closest(center_pos, x_vec, xp, yp)
    x_loc = [x_closest.x, x_closest.y]

    # 3. Compute Barycentric Coordinates (λ) and Gradients (dλ)
    x_A, y_A = p1
    x_B, y_B = p2
    x_C, y_C = p3
    denom = (y_B - y_C) * x_A + (y_C - y_A) * x_B + (y_A - y_B) * x_C
    
    # λ = [l0, l1, l2]
    l0 = ((y_B - y_C) * x_loc[1] + (x_C - x_B) * x_loc[2] + x_B * y_C - x_C * y_B) / denom
    l1 = ((y_C - y_A) * x_loc[1] + (x_A - x_C) * x_loc[2] + x_C * y_A - x_A * y_C) / denom
    l2 = ((y_A - y_B) * x_loc[1] + (x_B - x_A) * x_loc[2] + x_A * y_B - x_B * y_A) / denom
    lambda = [l0, l1, l2]

    # dλ = [dl0, dl1, dl2]
    dl0 = [y_B - y_C, x_C - x_B] / denom
    dl1 = [y_C - y_A, x_A - x_C] / denom
    dl2 = [y_A - y_B, x_B - x_A] / denom
    
    base_whitney = [
        l0 * dl1 - l1 * dl0,
        l1 * dl2 - l2 * dl1,
        l2 * dl0 - l0 * dl2
    ]

    # Align Whitney 1-forms with the global edge orientations for this triangle
    oriented_whitney = [edge_orientations[i] * base_whitney[i] for i in 1:3]

    # 5. Compute Higher Order Basis: λ^α * W
    indices = generate_multi_indices(poly_degree-1)
    basis_values = Dict{Int, Dict{MultiIndex, Vector{Float64}}}()
    
    for k in 1:3
        global_edge_idx = edge_indices[k]
        
        # Initialize inner dictionary for this edge
        basis_values[global_edge_idx] = Dict{MultiIndex, Vector{Float64}}()
        
        for alpha in indices
            poly_val = eval_monomial(lambda, alpha)
            basis_values[global_edge_idx][alpha] = poly_val * oriented_whitney[k]
        end
    end
    
    return basis_values, indices
end


In [ ]:
# Pick a triangle and a point
tri_idx = 2
points, edge_indices, edge_orientations = get_triangle(mesh, tri_idx)
poly_degree = 2
p = mesh.vertices.position[tri_idx]
# Compute basis functions λ^α * W where |α| = r-1
basis, indices = compute_higher_order_whitney_basis(mesh, tri_idx, p, poly_degree)

In [ ]:
basis[6]

In [ ]:
collect(values(basis[6]))

### Visualize the degree-2 edge basis — `plot_edge_basis`

For a chosen edge, plots each enriched basis function $\lambda^\alpha W_e$ (one subplot per multi-index $\alpha$) as a vector field over the triangle, evaluating `compute_higher_order_whitney_basis` on a grid. The next cell evaluates the basis at an edge midpoint.

In [ ]:
function plot_edge_basis(mesh::VoronoiMesh, triangle_idx::Int, local_edge_idx::Int, degree::Int)

    points, edge_indices, _ = get_triangle(mesh, triangle_idx)
    target_edge_global = edge_indices[local_edge_idx]
    
    p1, p2, p3 = points
    
    # 2. Create Grid over the Triangle
    # Bounding box with padding
    xs = [p1[1], p2[1], p3[1]]
    ys = [p1[2], p2[2], p3[2]]
    min_x, max_x = minimum(xs), maximum(xs)
    min_y, max_y = minimum(ys), maximum(ys)
    pad = 0.1 * max(max_x - min_x, max_y - min_y)
    
    grid_x = range(min_x - pad, max_x + pad, length=20)
    grid_y = range(min_y - pad, max_y + pad, length=20)
    alphas = generate_multi_indices(degree-1)
    n_plots = length(alphas)
    
    # Determine grid layout for subplots (approx square)
    cols = ceil(Int, sqrt(n_plots))
    rows = ceil(Int, n_plots / cols)
    
    fig = Figure(size=(500 * cols, 500 * rows))
    
    # 4. Iterate over each alpha and plot
    for (k, alpha) in enumerate(alphas)
        # Setup Axis
        row, col = divrem(k-1, cols)
        ax = Axis(fig[row+1, col+1], 
                  title = L"\alpha = %$alpha,\,\, \lambda_1^{%$(alpha[1])} \lambda_2^{%$(alpha[2])} \lambda_3^{%$(alpha[3])}W{e_{%$(local_edge_idx)}}" , 
                  titlesize = 22,
                  aspect = DataAspect())
        
        # Draw Triangle Boundary
        lines!(ax, [p1[1], p2[1], p3[1], p1[1]], [p1[2], p2[2], p3[2], p1[2]], color=:black)
        
        # Arrays for arrows
        arrow_x, arrow_y = Float64[], Float64[]
        arrow_u, arrow_v = Float64[], Float64[]
        
        # Compute vectors at each grid point
        for x in grid_x, y in grid_y
            pt = [x, y]
            
            # Compute ALL basis functions at this point
            basis_dict, indices = compute_higher_order_whitney_basis(mesh, triangle_idx, TensorsLite.Vec(x=x, y=y), degree)
            val = basis_dict[target_edge_global][alpha]
            
            push!(arrow_x, x)
            push!(arrow_y, y)
            push!(arrow_u, val[1])
            push!(arrow_v, val[2])
        end
        
        # Plot Arrows
        arrows!(ax, arrow_x, arrow_y, arrow_u, arrow_v, lengthscale=0.02, color=:blue)
        scatter!(ax, p1[1], p1[2], color = :orange, markersize = 12)
        scatter!(ax, p2[1], p2[2], color = :red, markersize = 12)
        lines!(ax, [p1[1], p2[1]], [p1[2], p2[2]], color = :red, linewidth=3)
    end
    
    Label(fig[0, :], "Basis Functions for Edge $local_edge_idx (Global: $target_edge_global)", fontsize=20)
    return fig
end

# Usage Example:
# Plot basis for Triangle 1, Local Edge  (p1-p2), Polynomial Degree 1
fig = plot_edge_basis(mesh, 2, 1, 2)
fig

In [ ]:
tri_idx = 2
points, edge_indices, edge_orientations = get_triangle(mesh, tri_idx)
p = 0.5(points[1] + points[2])
basis_dict, indices = compute_higher_order_whitney_basis(mesh, tri_idx, TensorsLite.Vec(x=p[1], y=p[2]), 2)



### Coarse vs. fine (medial) mesh

Builds the **fine** mesh by passing the coarse cell centres **and** the coarse edge points as generators to `VoronoiMesh(...; max_iter=0)` — a medial refinement of each coarse triangle. The plot overlays a fine vertex (red) on the corresponding coarse vertex (blue).

In [ ]:
coarse_mesh = VoronoiMesh("meshes/mesh1_regular.nc")
mesh = VoronoiMesh(vcat(coarse_mesh.cells.position, coarse_mesh.edges.position), 1.0, 1.0, max_iter=0)
fig = Figure(size=(600, 600))
ax = Axis(fig[1, 1])
plotmesh!(ax, mesh, color = :red)
points, edge_indices, edge_orientations = get_triangle(mesh, 2)
points_coarse, edge_indices_coarse, edge_orientations_coarse = get_triangle(coarse_mesh, 2)
scatter!(ax, points[1][1], points[1][2], color = :red, markersize = 22)
scatter!(ax, points_coarse[1][1], points_coarse[1][2], color = :blue, markersize = 22)

fig


# Mapping each coarse triangle to its fine sub-mesh

Each coarse Delaunay triangle is medially refined into 4 fine triangles (6 fine cells, 9 fine edges). The next cells build the bookkeeping that links a coarse triangle to the fine cells, fine **primal** edges, and fine **dual** edges that sit inside it — the local stencil used by every reconstruction and Hodge-star operator later in the notebook.

### Nearest-centroid coarse→fine map — `map_coarse_to_fine`

For each coarse triangle, finds its $k$ nearest fine triangles (by periodic-wrapped centroid distance) and returns, per coarse triangle, the fine-triangle ids, the unique fine-edge ids, and their orientations. This is the local stencil reused by every reconstruction and Hodge operator below. The next cell shows the result.

In [ ]:
# Nearest-centroid mapping: coarse triangle -> k nearest fine (Voronoi) triangles
# Uses triangle centroids with periodic wrapping to coarse centroid. Default k=4.
function map_coarse_to_fine(coarse_mesh::VoronoiMesh, fine_mesh::VoronoiMesh; k::Int=4)
    xp, yp = coarse_mesh.x_period, coarse_mesh.y_period
    coarse_centers = coarse_mesh.vertices.position
    fine_centers = fine_mesh.vertices.position
    nf = length(fine_centers)
    k_eff = min(k, nf)

    small_triangles = Dict{Int, Vector{Int}}()
    small_edges = Dict{Int, Vector{Int}}()
    small_edges_orientations = Dict{Int, Vector{Int}}()

    d2 = zeros(Float64, nf)

    for ci in 1:length(coarse_centers)
        c = coarse_centers[ci]
        # distances to all fine centers with periodic wrapping
        for j in 1:nf
            v = closest(c, fine_centers[j], xp, yp)
            dv = v - c
            d2[j] = dot(dv, dv)
        end
        idxs = partialsortperm(d2, 1:k_eff)
        small_triangles[ci] = collect(idxs)

        # gather fine edges for those triangles
        edge_ids = Int[]
        edge_orientations = Int[]
        for fid in idxs
            _, fine_edges, fine_edges_orientations = get_triangle(fine_mesh, fid)
            append!(edge_ids, fine_edges)
            append!(edge_orientations, fine_edges_orientations)
        end
        small_edges[ci] = unique(edge_ids)
        small_edges_orientations[ci] = edge_orientations
    end

    return small_triangles, small_edges, small_edges_orientations
end

In [ ]:
coarse_mesh = VoronoiMesh("meshes/mesh1_regular.nc")
fine_mesh = VoronoiMesh(vcat(coarse_mesh.cells.position, coarse_mesh.edges.position), 1.0, 1.0, max_iter=0)
small_triangles, small_edges, small_edges_orientations = map_coarse_to_fine(coarse_mesh, fine_mesh; k=4)
small_triangles

### Plot the fine children of a coarse triangle — `plot_children_nearest`

Draws the $k$ nearest fine triangles of a coarse triangle (filled), with the fine primal edges as arrows, all wrapped to the coarse circumcenter so periodic seams render continuously.

In [ ]:
coarse_mesh = VoronoiMesh("meshes/mesh1_regular.nc")
fine_mesh = VoronoiMesh(vcat(coarse_mesh.cells.position, coarse_mesh.edges.position), 1.0, 1.0, max_iter=0)

function plot_children_nearest(coarse_mesh::VoronoiMesh, fine_mesh::VoronoiMesh, coarse_triangle_idx::Int; show_coarse=true, color=:lightblue)
    small_triangles, small_edges, small_edges_orientations = map_coarse_to_fine(coarse_mesh, fine_mesh; k=4)
    fine_ids = small_triangles[coarse_triangle_idx]
    edge_ids = small_edges[coarse_triangle_idx]
    edge_orients = small_edges_orientations[coarse_triangle_idx]
    
    xp, yp = coarse_mesh.x_period, coarse_mesh.y_period
    triangle_center_position = coarse_mesh.vertices.position[coarse_triangle_idx]
    cpts, _, _ = get_triangle(coarse_mesh, coarse_triangle_idx)
    
    cxs = Float64[]; cys = Float64[]
    for c in cpts
        v = closest(triangle_center_position, TensorsLite.Vec(x=c[1], y=c[2]), xp, yp)
        push!(cxs, v.x); push!(cys, v.y)
    end

    fig = Figure(); ax = Axis(fig[1, 1], aspect=DataAspect())

    if show_coarse
        lines!(ax, [cxs; cxs[1]], [cys; cys[1]], color=:black, linewidth=2)
    end

    # Draw fine triangles
    for fid in fine_ids
        pts, _, _ = get_triangle(fine_mesh, fid)
        xs = Float64[]; ys = Float64[]
        for p in pts
            v = closest(triangle_center_position, TensorsLite.Vec(x=p[1], y=p[2]), xp, yp)
            push!(xs, v.x); push!(ys, v.y)
        end
        poly!(ax, Point2f.([xs; xs[1]], [ys; ys[1]]); color=color, alpha=0.25)
        lines!(ax, [xs; xs[1]], [ys; ys[1]], color=:blue, linewidth=1)
        scatter!(ax, xs, ys, color = :red, markersize = 22)
    end

    # Draw oriented fine-edge arrows using edge vertex connectivity
    arrow_x = Float64[]; arrow_y = Float64[]; arrow_u = Float64[]; arrow_v = Float64[]
    for (eid, orient) in zip(edge_ids, edge_orients)
        v1, v2 = fine_mesh.edges.cells[eid]
        p1 = closest(triangle_center_position, fine_mesh.cells.position[v1], xp, yp)
        p2 = closest(triangle_center_position, fine_mesh.cells.position[v2], xp, yp)
        
        start_pt, end_pt = p1, p2 #no orientation
        #start_pt, end_pt = orient >= 0 ? (p1, p2) : (p2, p1)#with orientation

        push!(arrow_x, start_pt.x)
        push!(arrow_y, start_pt.y)
        push!(arrow_u, 0.5 * (end_pt.x - start_pt.x))
        push!(arrow_v, 0.5 * (end_pt.y - start_pt.y))
    end
    arrows!(ax, arrow_x, arrow_y, arrow_u, arrow_v; color=:green, linewidth=1, arrowsize=22, lengthscale=1.0)

    fig
end

# Example usage:
plot_children_nearest(coarse_mesh, fine_mesh, 2)

### Dual edges of a coarse triangle

- `dual_edge_segment` — the dual (Voronoi) edge of a fine edge, with endpoints wrapped near an anchor;
- `dual_edges_for_coarse_triangle` — gathers all fine dual edges under a coarse triangle with consistent **orientations** (sign of the primal×dual cross product) and the dual-vertex indices;
- `plot_dual_edges_for_coarse_triangle` — draws them as oriented arrows.

The following two cells demo the plot and unpack the returned named tuple.

In [ ]:
# Dual edge segment (wrapped near a chosen anchor, e.g., coarse triangle center)
function dual_edge_segment(fine_mesh::VoronoiMesh, edge_idx::Int, anchor::TensorsLite.Vec)
    xp, yp = fine_mesh.x_period, fine_mesh.y_period
    v1, v2 = fine_mesh.edges.vertices[edge_idx]             # dual-edge endpoints (dual cells)
    p1 = closest(anchor, fine_mesh.vertices.position[v1], xp, yp)
    p2 = closest(anchor, fine_mesh.vertices.position[v2], xp, yp)
    return p1, p2, v1, v2
end

# Collect dual edges associated to a coarse triangle (
function dual_edges_for_coarse_triangle(coarse_mesh::VoronoiMesh, fine_mesh::VoronoiMesh, coarse_triangle_idx::Union{Int32, Int64}; k::Int=4)
    small_tris, small_edges, small_orients = map_coarse_to_fine(coarse_mesh, fine_mesh; k=k)
    edge_ids      = small_edges[coarse_triangle_idx]

    xp, yp = coarse_mesh.x_period, coarse_mesh.y_period
    anchor = coarse_mesh.vertices.position[coarse_triangle_idx]

    # coarse triangle wrapped near anchor
    cpts, _, _ = get_triangle(coarse_mesh, coarse_triangle_idx)
    coarse_pts_wrapped = [closest(anchor, TensorsLite.Vec(x=p[1], y=p[2]), xp, yp) for p in cpts]

    # dual-edge segments wrapped near the same anchor
    segments = []
    orientations = []
    dual_vertices_idx = []
    for eid in edge_ids
        dual_p1, dual_p2, dual_p1_idx, dual_p2_idx = dual_edge_segment(fine_mesh, eid, anchor)
        v1, v2 = fine_mesh.edges.cells[eid]
        primal_p1 = closest(anchor, fine_mesh.cells.position[v1], xp, yp)
        primal_p2 =  closest(anchor, fine_mesh.cells.position[v2], xp, yp)

        primal_vec = primal_p2 - primal_p1
        dual_vec = dual_p2 - dual_p1
        dual_orientation = primal_vec.x * dual_vec.y - primal_vec.y * dual_vec.x > 0 ? 1 : -1

        push!(segments, (dual_p1, dual_p2))
        push!(orientations, dual_orientation)
        push!(dual_vertices_idx, (dual_p1_idx, dual_p2_idx))
    end

    dual_vertices_by_edge_id = Dict(eid => seg for (eid, seg) in zip(edge_ids, segments))
    dual_edge_orientations_by_edge_id = Dict(eid => orient for (eid, orient) in zip(edge_ids, orientations))
    dual_vertices_idx_by_edge_id = Dict(eid => idx for (eid, idx) in zip(edge_ids, dual_vertices_idx))

    return (edge_ids=edge_ids,
            dual_vertices = dual_vertices_by_edge_id,
            dual_edge_orientations = dual_edge_orientations_by_edge_id,
            dual_vertices_idx = dual_vertices_idx_by_edge_id)             # each segment is (p1::Vec, p2::Vec)
end

function plot_dual_edges_for_coarse_triangle(coarse_mesh, fine_mesh, coarse_triangle_idx; k=4, color=:green)
    data = dual_edges_for_coarse_triangle(coarse_mesh, fine_mesh, coarse_triangle_idx; k=k)
    xp, yp = coarse_mesh.x_period, coarse_mesh.y_period
    coarse_pts, _, _ = get_triangle(coarse_mesh, coarse_triangle_idx)
    cxs = [p[1] for p in coarse_pts]; cys = [p[2] for p in coarse_pts]

    fig = Figure(); ax = Axis(fig[1,1], aspect=DataAspect())
    lines!(ax, [cxs; cxs[1]], [cys; cys[1]], color=:black, linewidth=2)

    for ((p1, p2), orient) in zip(collect(values(data.dual_vertices)), values(data.dual_edge_orientations))
        start_pt, end_pt = orient >= 0 ? (p1, p2) : (p2, p1)
        arrows!(ax, [start_pt.x], [start_pt.y], [end_pt.x - start_pt.x], [end_pt.y - start_pt.y];
                color=color, linewidth=1, arrowsize=18, lengthscale=1.0)
    end
    fig
end

In [ ]:
plot_dual_edges_for_coarse_triangle(coarse_mesh, fine_mesh, 2)

In [ ]:
edge_idx, dual_vertices, edge_orientations, dual_vertices_idx = dual_edges_for_coarse_triangle(coarse_mesh, fine_mesh, 1)


### Dual edges for every coarse triangle — `dual_edges_dict`

Applies `dual_edges_for_coarse_triangle` to all coarse triangles, returning a dictionary of dual-edge data keyed by coarse triangle. The cell after re-runs `map_coarse_to_fine` for quick inspection.

In [ ]:
# Dictionary over all coarse triangles
function dual_edges_dict(coarse_mesh::VoronoiMesh, fine_mesh::VoronoiMesh; k::Int=4)
    d = Dict{Int, NamedTuple}()
    n_tri = length(coarse_mesh.vertices.position)
    for ci in 1:n_tri
        d[ci] = dual_edges_for_coarse_triangle(coarse_mesh, fine_mesh, ci; k=k)
    end
    return d
end

dual_data = dual_edges_dict(coarse_mesh, fine_mesh; k=4)[5].dual_vertices
# dual_data[1]  # data for coarse triangle 1
# plot_dual_edges_for_coarse_triangle(coarse_mesh, fine_mesh, 1)


In [ ]:
small_triangles, small_edges, small_orients = map_coarse_to_fine(coarse_mesh, fine_mesh; k=4)

### Visualize a coarse-edge neighborhood — `plot_coarse_edge_neighborhood`

Draws the two coarse triangles sharing a coarse edge together with their fine triangles and dual edges — the geometry behind any $1\to2$ or Hodge-star stencil that straddles a coarse edge.

In [ ]:
function plot_coarse_edge_neighborhood(coarse_mesh::VoronoiMesh,
                                        fine_mesh::VoronoiMesh,
                                        coarse_edge_idx::Union{Int32, Int64}; k::Int=4)
    xp, yp = coarse_mesh.x_period, coarse_mesh.y_period
    t1, t2 = coarse_mesh.edges.vertices[coarse_edge_idx]     
    coarse_tris = [t1, t2]

    small_tris, small_edges, small_orients = map_coarse_to_fine(coarse_mesh, fine_mesh; k=k)

    fig = Figure(size=(1000, 1000)); ax = Axis(fig[1,1], aspect=DataAspect())
    coarse_colors = [:lightblue, :lightgreen]

    # Plot each adjacent coarse triangle and its fine/dual data
    for (cti, ccol) in zip(coarse_tris, coarse_colors)
        anchor = coarse_mesh.vertices.position[cti]
        cpts, _, _ = get_triangle(coarse_mesh, cti)
        cx = Float64[]; cy = Float64[]
        for p in cpts
            v = closest(anchor, TensorsLite.Vec(x=p[1], y=p[2]), xp, yp)
            push!(cx, v.x); push!(cy, v.y)
        end
        poly!(ax, [cx; cx[1]], [cy; cy[1]]; color=ccol, alpha=0.12, strokecolor=:black, strokewidth=3)

        # Fine triangles under this coarse triangle
        fine_ids = small_tris[cti]
        for fid in fine_ids
            pts, fine_eids, fine_eorients = get_triangle(fine_mesh, fid)
            xs = Float64[]; ys = Float64[]
            for p in pts
                v = closest(anchor, TensorsLite.Vec(x=p[1], y=p[2]), xp, yp)
                push!(xs, v.x); push!(ys, v.y)
            end
            poly!(ax, [xs; xs[1]], [ys; ys[1]]; color=:gray90, alpha=0.35, strokecolor=:blue, strokewidth=3)
            scatter!(ax, xs, ys; color=:red, markersize=12)

            # Oriented fine edges as arrows
            for (eid, eor) in zip(fine_eids, fine_eorients)
                v1, v2 = fine_mesh.edges.cells[eid]
                p1 = closest(anchor, fine_mesh.cells.position[v1], xp, yp)
                p2 = closest(anchor, fine_mesh.cells.position[v2], xp, yp)
                s, e = p1, p2
                arrows!(ax, [s.x], [s.y], [0.5 * (e.x - s.x)], [0.5 * (e.y - s.y)];
                color=:cyan, linewidth=2, arrowsize=20, lengthscale=1.0)
            end
        end

        # Dual edges for this coarse triangle
        dual = dual_edges_for_coarse_triangle(coarse_mesh, fine_mesh, cti; k=k)
        for ((p1, p2), orient) in zip(collect(values(dual.dual_vertices)), values(dual.dual_edge_orientations))
            s, e = orient >= 0 ? (p1, p2) : (p2, p1)
            arrows!(ax, [s.x], [s.y], [e.x - s.x], [e.y - s.y];
            color=:orange, linewidth=1, arrowsize=12, lengthscale=1.0)
        end
    end

    fig
end

# Usage example:

coarse_mesh = VoronoiMesh("meshes/mesh1.nc")
fine_mesh = VoronoiMesh(vcat(coarse_mesh.cells.position, coarse_mesh.edges.position), 1.0, 1.0, max_iter=0)
fig = plot_coarse_edge_neighborhood(coarse_mesh, fine_mesh, 3; k=4)
fig

### Bundle all per-coarse-triangle data — `coarse_triangle_data_dict`

Collects everything needed per coarse triangle into one structure: the (wrapped) coarse vertices, its fine triangles with their edge ids/orientations/segments, the unique fine primal edges, and the fine dual edges. The two cells after it build and inspect this dictionary.

In [ ]:
function coarse_triangle_data_dict(coarse_mesh::VoronoiMesh,
                                   fine_mesh::VoronoiMesh; k::Int=4)
    xp, yp = coarse_mesh.x_period, coarse_mesh.y_period
    small_tris, small_edges, _ = map_coarse_to_fine(coarse_mesh, fine_mesh; k=k)

    data = Dict{Int, NamedTuple}()

    for ci in 1:length(coarse_mesh.vertices.position)
        anchor = coarse_mesh.vertices.position[ci]

        # coarse triangle wrapped
        cpts, _, _ = get_triangle(coarse_mesh, ci)
        coarse_pts_wrapped = [closest(anchor, TensorsLite.Vec(x=p[1], y=p[2]), xp, yp) for p in cpts]

        # fine triangles mapped to this coarse triangle
        fine_ids = small_tris[ci]
        fine_tris = NamedTuple[]
        fine_edges_flat = NamedTuple[]
        seen_edge = Dict{Int,Bool}()

        for fid in fine_ids
            pts, eids, eors = get_triangle(fine_mesh, fid)
            verts_wrapped = [closest(anchor, TensorsLite.Vec(x=p[1], y=p[2]), xp, yp) for p in pts]

            # oriented edge segments for this fine triangle
            edge_segments = Vector{Tuple{Any,Any}}()
            for (eid, eor) in zip(eids, eors)
                v1, v2 = fine_mesh.edges.vertices[eid]
                p1 = closest(anchor, fine_mesh.vertices.position[v1], xp, yp)
                p2 = closest(anchor, fine_mesh.vertices.position[v2], xp, yp)
                s, e = p1, p2
                push!(edge_segments, (s, e))

                # add to flat list once
                if !haskey(seen_edge, eid)
                    seen_edge[eid] = true
                    push!(fine_edges_flat, (edge_id=eid, orient=eor, p1=p1, p2=p2))
                end
            end

            push!(fine_tris, (id=fid,
                              vertices=verts_wrapped,
                              edge_ids=eids,
                              edge_orientations=eors,
                              edge_segments=edge_segments))
        end

        # dual edges for this coarse triangle (uses your existing helper)
        dual = dual_edges_for_coarse_triangle(coarse_mesh, fine_mesh, ci; k=k)

        data[ci] = (
            coarse_pts = coarse_pts_wrapped,
            anchor = anchor,
            fine_triangles = fine_tris,
            fine_edges = fine_edges_flat,  # unique fine edges with orientation/endpoints
            dual_edges = (
                edge_ids = dual.edge_ids,
                orientations = dual.dual_edge_orientations,
                vertices = dual.dual_vertices
            )
        )
    end

    return data
end

# Example:
# d = coarse_triangle_data_dict(coarse_mesh, fine_mesh; k=4)
# d[1].fine_triangles         # all fine tris mapped to coarse tri 1
# d[1].fine_edges             # unique fine edges with orientation/endpoints
# d[1].dual_edges.vertices    # dual edge endpoints

In [ ]:
data = coarse_triangle_data_dict(coarse_mesh, fine_mesh; k=4)

In [ ]:
data[1].fine_triangles

### Whitney LSQ matrix and reconstruction helpers

The core machinery for the higher-order 1-form reconstruction $I_h^1$:
- `build_whitney_matrix_for_coarse_triangle` — assembles $A_1$ with entry $\int_{\text{fine edge }i}\lambda^\alpha W_e\cdot d\ell$ (rows = fine edges, columns = pairs $(e,\alpha)$), wrapping each fine edge to its own primal–dual point so periodic boundaries use the short path (matching `discretize_vector_field_dual`);
- `build_whitney_matrices_for_mesh` — builds $A_1$ for every coarse triangle (optionally multithreaded);
- `independent_columns` — rank-revealing QR (`pqrfact`) dropping linearly dependent columns so the LSQ solve is well posed (9×9 → effective 9×8);
- `combined_whitney_field` — evaluates the reconstructed field $\sum c_{e,\alpha}\lambda^\alpha W_e$ at a point;
- `combined_whitney_curl`, `whitney_curl_basis_at`, `grad_monomial` — its closed-form curl.

In [ ]:
function build_whitney_matrix_for_coarse_triangle(coarse_mesh::VoronoiMesh,
        fine_mesh::VoronoiMesh,
        coarse_triangle_idx::Int,
        small_edges=nothing, small_orients=nothing; degree=2, k=4, nq=5)
    xp, yp = coarse_mesh.x_period, coarse_mesh.y_period

    # coarse triangle info
    _, coarse_edge_ids, _ = get_triangle(coarse_mesh, coarse_triangle_idx)
    alphas = generate_multi_indices(degree - 1)  # e.g., degree=2 ⇒ 3 alphas
    cols_key = [(eid, alpha) for eid in coarse_edge_ids for alpha in alphas]
    ncols = length(cols_key)

    # fine edges under this coarse triangle (ids + orientations) from map_coarse_to_fine
    if small_edges === nothing || small_orients === nothing
        _, small_edges, small_orients = map_coarse_to_fine(coarse_mesh, fine_mesh)
    end
    edge_ids = small_edges[coarse_triangle_idx]
    edge_orients = small_orients[coarse_triangle_idx]
    rows_edges = collect(zip(edge_ids, edge_orients))
    nrows = length(rows_edges)

    A = zeros(Float64, nrows, ncols)
    points, weights = gaussian_quadrature_points_and_weights(nq)

    # loop over fine edges (rows)
    for (row_idx, (feid, forient)) in enumerate(rows_edges)
        v1, v2 = fine_mesh.edges.cells[feid]
        p1_raw = fine_mesh.cells.position[v1]
        p2_raw = fine_mesh.cells.position[v2]
        # Wrap relative to the fine edge's primal-dual intersection point.
        # mesh.edges.position[feid] lies ON the c1–c2 segment (between c1 and c2),
        # so this always selects the SHORT path across any periodic boundary,
        # matching the convention used by discretize_vector_field_dual.
        edge_ref = fine_mesh.edges.position[feid]
        p1 = closest(edge_ref, p1_raw, xp, yp)
        p2 = closest(edge_ref, p2_raw, xp, yp)

        p1_vec = [p1[1], p1[2]]
        p2_vec = [p2[1], p2[2]]
        edge_vector = p2_vec .- p1_vec

        # integrate all basis columns over this fine edge
        for (xi, wi) in zip(points, weights)
            t = (xi + 1) / 2
            p = p1_vec .+ t .* edge_vector
            x_t = TensorsLite.Vec(x=p[1], y=p[2])
            basis_dict, _ = compute_higher_order_whitney_basis(coarse_mesh, coarse_triangle_idx, x_t, degree)

            for (col_idx, (ceid, alpha)) in enumerate(cols_key)
                A[row_idx, col_idx] += wi * dot(basis_dict[ceid][alpha], edge_vector)
            end
        end

        A[row_idx, :] .*= (forient / 2)
    end

    return A, rows_edges, cols_key
end

function build_whitney_matrices_for_mesh(coarse_mesh::VoronoiMesh,
    fine_mesh::VoronoiMesh; degree=2, k=4, nq=5, use_threads=false)
    n_tris = coarse_mesh.vertices.n
    matrices = Vector{Matrix{Float64}}(undef, n_tris)
    rows_edges_list = Vector{Any}(undef, n_tris)
    cols_key_list = Vector{Any}(undef, n_tris)
    _, small_edges, small_orients = map_coarse_to_fine(coarse_mesh, fine_mesh)

    if use_threads
        Threads.@threads for tri in 1:n_tris
            A, rows_edges, cols_key = build_whitney_matrix_for_coarse_triangle(
            coarse_mesh, fine_mesh, tri, small_edges, small_orients; degree=degree, k=k, nq=nq
            )
            matrices[tri] = A
            rows_edges_list[tri] = rows_edges
            cols_key_list[tri] = cols_key
        end
    else
        for tri in 1:n_tris
            A, rows_edges, cols_key = build_whitney_matrix_for_coarse_triangle(
            coarse_mesh, fine_mesh, tri, small_edges, small_orients; degree=degree, k=k, nq=nq
            )
            matrices[tri] = A
            rows_edges_list[tri] = rows_edges
            cols_key_list[tri] = cols_key
        end
    end
    return (A=matrices, rows_edges=rows_edges_list, cols_key=cols_key_list)
end

# Rank-revealing decomposition
function independent_columns(A; rtol=1e-12)
    F = pqrfact(A, rank=8)
    diagR = diag(F.R)
    tol = rtol * maximum(abs.(diagR))
    r = count(abs.(diagR) .> tol)
    cols = F.p[1:r]
    return r, cols, A[:, cols]
end

# evaluate the reconstructed higher-order Whitney combination at a point
function combined_whitney_field(mesh, tri_idx, coeffs, cols_key, degree)
    return x -> begin
        x_vec = TensorsLite.Vec(x=x[1], y=x[2])
        basis_dict, _ = compute_higher_order_whitney_basis(mesh, tri_idx, x_vec, degree)
        sum = zeros(2)
        for (c, (eid, alpha)) in zip(coeffs, cols_key)
            sum .+= c .* basis_dict[eid][alpha]
        end
        sum
    end
end

# gradient of a barycentric monomial  λ^α  (used for analytical curl below)
function grad_monomial(lambda, alpha, dl)
    gx, gy = 0.0, 0.0
    for i in 1:3
        if alpha[i] > 0
            factor = eval_monomial(lambda, alpha) / lambda[i]
            gx += alpha[i] * factor * dl[i][1]
            gy += alpha[i] * factor * dl[i][2]
        end
    end
    return gx, gy
end

# curl(λ^α W_e) for one Whitney column;  W_e = λ_a ∇λ_b − λ_b ∇λ_a,  curl(W_e) = 2 dλ_a ∧ dλ_b
function whitney_curl_basis_at(mesh, tri_idx, global_edge_idx, alpha, p, degree)
    _, edge_indices, edge_orientations = get_triangle(mesh, tri_idx)
    k = findfirst(==(global_edge_idx), edge_indices)
    k === nothing && error("edge $global_edge_idx not in coarse triangle $tri_idx")
    orient = edge_orientations[k]
    edge_pairs = ((1, 2), (2, 3), (3, 1))   # local Whitney edge k uses (λ_{a}, λ_{b})
    ia, ib = edge_pairs[k]

    center = mesh.vertices.position[tri_idx]
    xp, yp = mesh.x_period, mesh.y_period
    cpts, _, _ = get_triangle(mesh, tri_idx)
    p1 = closest(center, TensorsLite.Vec(x=cpts[1][1], y=cpts[1][2]), xp, yp)
    p2 = closest(center, TensorsLite.Vec(x=cpts[2][1], y=cpts[2][2]), xp, yp)
    p3 = closest(center, TensorsLite.Vec(x=cpts[3][1], y=cpts[3][2]), xp, yp)
    x_c = closest(center, TensorsLite.Vec(x=p[1], y=p[2]), xp, yp)
    xA, yA = p1.x, p1.y; xB, yB = p2.x, p2.y; xC, yC = p3.x, p3.y
    denom = (yB-yC)*xA + (yC-yA)*xB + (yA-yB)*xC

    l = [
        ((yB-yC)*x_c.x + (xC-xB)*x_c.y + xB*yC - xC*yB) / denom,
        ((yC-yA)*x_c.x + (xA-xC)*x_c.y + xC*yA - xA*yC) / denom,
        ((yA-yB)*x_c.x + (xB-xA)*x_c.y + xA*yB - xB*yA) / denom,
    ]
    dl = [
        [yB-yC, xC-xB] ./ denom,
        [yC-yA, xA-xC] ./ denom,
        [yA-yB, xB-xA] ./ denom,
    ]

    Wa, Wb = dl[ib], dl[ia]
    Wx = l[ia]*Wa[1] - l[ib]*Wb[1]
    Wy = l[ia]*Wa[2] - l[ib]*Wb[2]
    poly = eval_monomial(l, alpha)
    gx, gy = grad_monomial(l, alpha, dl)
    curl_W = 2.0 * (dl[ia][1]*dl[ib][2] - dl[ia][2]*dl[ib][1])   # 2 dλ_a ∧ dλ_b
    return orient * (gx * Wy - gy * Wx + poly * curl_W)
end

# analytical curl of  I_h¹ ω  = Σ c_{e,α} λ^α W_e   (scalar 2-form coefficient)
function combined_whitney_curl(mesh, tri_idx, coeffs, cols_key, degree)
    return p -> sum(c * whitney_curl_basis_at(mesh, tri_idx, eid, alpha, p, degree)
                    for (c, (eid, alpha)) in zip(coeffs, cols_key))
end

### Sanity check: reconstruct a linear field on one triangle

Discretizes $v(x)=(x_2-2x_1,\;x_1-2)$ on the fine mesh, assembles the Whitney matrix for one coarse triangle, solves the rank-revealing LSQ system, and verifies the reconstruction reproduces $v$ at the circumcenter. Uses `discretize_vector_field_dual`, `build_whitney_matrix_for_coarse_triangle`, `independent_columns`, and `combined_whitney_field`.

In [ ]:
# Constant field test on a coarse triangle with rank-revealing solve
coarse_mesh = VoronoiMesh("meshes/mesh2.nc")
fine_mesh = VoronoiMesh(vcat(coarse_mesh.cells.position, coarse_mesh.edges.position),
                        coarse_mesh.x_period, coarse_mesh.y_period; max_iter=0)

vector_field(x) = [x[2] - 2*x[1], x[1] - 2]
fine_v = discretize_vector_field_dual(fine_mesh, vector_field)

# Assemble Whitney matrix for one coarse triangle
coarse_tri = 1
A, rows_edges, cols_key = build_whitney_matrix_for_coarse_triangle(coarse_mesh, fine_mesh, coarse_tri; degree=2, k=4, nq=3)
edge_ids = first.(rows_edges)
edge_orients = last.(rows_edges)

# Right-hand side: fine edge integrals (aligned with row orientations)
rhs = fine_v.values[edge_ids].*edge_orients

rankA, col_keep, A_ind = independent_columns(A; rtol=1e-6)
println("rank(A) = $rankA / $(size(A,2)), condition number: $(cond(A_ind)), keeping columns: ", col_keep)

coeffs = A_ind \ rhs
b = A_ind * coeffs
println("max |A*coeffs - rhs| = ", maximum(abs.(b .- rhs)))

approximation = combined_whitney_field(coarse_mesh, coarse_tri, coeffs, cols_key[col_keep], 2)
p = coarse_mesh.vertices.position[coarse_tri]
println("Error: $(maximum(abs.(approximation(p) - vector_field(p))))")


### Commutation with $d$ for a discrete 0-form

A discrete 0-form lives on the **coarse triangle nodes** (cell centres):
$\omega_0 \in C^0_p(\text{coarse\_mesh})$, one value per vertex of each coarse Delaunay triangle.

Both reconstruction operators are then closed-form on a coarse triangle — no LSQ:

- $R_0(\omega_0)(p) \;=\; \sum_{i=1}^{3}\omega_0[v_i]\,\lambda_i(p)$ (barycentric)
- $R_1(\bar d\,\omega_0)(p) \;=\; \sum_{e}(\bar d\,\omega_0)[e]\,W_e(p)$ (linear Whitney)

The commutation $d\circ R_0 = R_1\circ\bar d$ is an **algebraic identity**:
$$
dR_0\,\omega_0 \;=\; \sum_i \omega_0[v_i]\,d\lambda_i \;=\; R_1(\bar d\,\omega_0),
$$
which one proves on a single triangle using $\sum_i\lambda_i = 1$ and $\sum_i d\lambda_i = 0$. So the test below should pass at machine precision for **any** smooth $\tilde\omega$ used to sample $\omega_0$ and at **any** point of the coarse triangle — no convergence, no polynomial restriction.

In [ ]:
# ── Linear Whitney commutation:  d ∘ R₀  =  R₁ ∘ d̄  on a coarse triangle ──
#
# ω₀ ∈ C⁰_p(coarse_mesh):   ONE value per coarse triangle node (= cell centre).
# R₀(ω₀)(p)   = Σᵢ ω₀[vᵢ] λᵢ(p)            barycentric — no LSQ, 3 dofs
# R₁(d̄ω₀)(p)  = Σₑ (d̄ω₀)[e] Wₑ(p)         linear Whitney 1-form, 3 edge dofs
#
# By the algebraic identity dR₀ = R₁d̄, both sides equal Σᵢ ω₀[vᵢ] ∇λᵢ — a single
# constant vector on the triangle.  The check is exact for any ω̃ (no quadrature,
# no convergence, no polynomial restriction).

# Sample 0-form (any smooth function; the identity holds for arbitrary ω₀)
ω̃(x) = sin(2π*x[1])*cos(2π*x[2]) + 0.3*x[1] - 0.7*x[2]^2

# Discrete 0-cochain at coarse cell centres   (= coarse triangle nodes)
ω₀  = primal_cochain([ω̃([p.x, p.y]) for p in coarse_mesh.cells.position], 0)

# Discrete d̄ : C⁰_p → C¹_p     (d0_matrix_primal × ω₀.values, via Cochain dispatch)
dω₀ = d(coarse_mesh, ω₀)

# Pick a coarse triangle
coarse_tri = 1
pts, _, _  = get_triangle(coarse_mesh, coarse_tri)

# ── LHS:  d(R₀ ω₀) = Σᵢ ω₀[vᵢ] ∇λᵢ   (constant 2-vector on the triangle) ──
xA, yA = pts[1];  xB, yB = pts[2];  xC, yC = pts[3]
denom  = (yB - yC)*xA + (yC - yA)*xB + (yA - yB)*xC
∇λ = (
    [yB - yC, xC - xB] ./ denom,
    [yC - yA, xA - xC] ./ denom,
    [yA - yB, xB - xA] ./ denom,
)
ω_v = ω₀.values[coarse_mesh.vertices.cells[coarse_tri]]
LHS = ω_v[1] .* ∇λ[1] .+ ω_v[2] .* ∇λ[2] .+ ω_v[3] .* ∇λ[3]

# ── RHS:  R₁(d̄ω₀)  via the existing linear Whitney interpolant ──
RHS = whitney_interpolation_on_triangle(coarse_mesh, coarse_tri, dω₀)

println("LHS  d(R₀ω₀)         = ", LHS)
println("RHS  R₁(d̄ω₀)         = ", RHS)
println("|LHS − RHS|∞          = ", maximum(abs.(LHS .- RHS)))

### Higher-order commutation: degree = 2

Both operators become LSQ fits on a fine-mesh patch around each coarse triangle:

- $R_0^{(2)}$: fit a $P_2\Lambda^0$ polynomial — basis $\{\lambda^\alpha : |\alpha|=2\}$ (6 functions) — to the fine cell values in the patch. On a regular `mesh*_regular.nc` this is *exact* P₂ Lagrange interpolation at the 3 vertices + 3 edge midpoints (6 nodes, 6 dof — square).
- $R_1^{(2)}$: the existing rank-revealing fit in $P_2^-\Lambda^1$ (rank 8, 9 fine-edge equations).

On the polynomial subspace ($\tilde\omega\in P_2$) both sides reproduce $\nabla\tilde\omega$ *exactly*, so the commutation error must hit machine precision. On a non-polynomial input it is a real question — the LSQ DOFs of $R_0$ and $R_1$ are not the canonical FEEC DOFs, so closure of the diagram is not automatic.

In [ ]:
# ── Higher-order commutation:  d ∘ R₀^(2)  =  R₁^(2) ∘ d̄  on a coarse triangle ──

# R₀^(2): fit a P_2 Λ⁰ polynomial  (basis {λ^α : |α|=2}, 6 functions)
function build_lagrange0_matrix_for_coarse_triangle(coarse_mesh, fine_mesh, coarse_tri; degree=2, k=4)
    xp, yp     = coarse_mesh.x_period, coarse_mesh.y_period
    anchor     = coarse_mesh.vertices.position[coarse_tri]
    cpts, _, _ = get_triangle(coarse_mesh, coarse_tri)
    xA, yA = cpts[1]; xB, yB = cpts[2]; xC, yC = cpts[3]
    denom  = (yB-yC)*xA + (yC-yA)*xB + (yA-yB)*xC

    alphas           = generate_multi_indices(degree)
    small_tris, _, _ = map_coarse_to_fine(coarse_mesh, fine_mesh; k=k)
    cell_ids         = unique(reduce(vcat,
        [collect(fine_mesh.vertices.cells[fid]) for fid in small_tris[coarse_tri]]))

    A = zeros(length(cell_ids), length(alphas))
    for (i, fcid) in enumerate(cell_ids)
        p  = closest(anchor, fine_mesh.cells.position[fcid], xp, yp)
        l0 = ((yB-yC)*p[1] + (xC-xB)*p[2] + xB*yC - xC*yB) / denom
        l1 = ((yC-yA)*p[1] + (xA-xC)*p[2] + xC*yA - xA*yC) / denom
        l2 = ((yA-yB)*p[1] + (xB-xA)*p[2] + xA*yB - xB*yA) / denom
        for (j, α) in enumerate(alphas)
            A[i, j] = l0^α[1] * l1^α[2] * l2^α[3]
        end
    end
    return A, cell_ids, alphas, (xA,yA,xB,yB,xC,yC,denom)
end

# d(R₀ω₀)(p)  =  Σ_α c_α ∇(λ^α)(p)        (closed form, no quadrature)
function grad_R0_at(p, coeffs_0, alphas, geom)
    xA,yA,xB,yB,xC,yC,denom = geom
    l0 = ((yB-yC)*p[1] + (xC-xB)*p[2] + xB*yC - xC*yB) / denom
    l1 = ((yC-yA)*p[1] + (xA-xC)*p[2] + xC*yA - xA*yC) / denom
    l2 = ((yA-yB)*p[1] + (xB-xA)*p[2] + xA*yB - xB*yA) / denom
    λ  = (l0, l1, l2)
    ∇λ = ([yB-yC, xC-xB] ./ denom, [yC-yA, xA-xC] ./ denom, [yA-yB, xB-xA] ./ denom)

    result = [0.0, 0.0]
    for (c, α) in zip(coeffs_0, alphas)
        for i in 1:3
            α[i] >= 1 || continue
            pow      = ntuple(j -> j == i ? α[j] - 1 : α[j], 3)
            poly_val = λ[1]^pow[1] * λ[2]^pow[2] * λ[3]^pow[3]
            result .+= c * α[i] * poly_val .* ∇λ[i]
        end
    end
    return result
end

# Driver: returns (max commutation error, info NamedTuple)
function commutation_error_deg2(coarse_mesh, fine_mesh, coarse_tri, ω̃; k=4, nq=3, rtol=1e-6)
    # R₀
    A0, cell_ids, alphas_0, geom = build_lagrange0_matrix_for_coarse_triangle(
        coarse_mesh, fine_mesh, coarse_tri; degree=2, k=k)
    ω₀       = primal_cochain([ω̃([p.x, p.y]) for p in fine_mesh.cells.position], 0)
    coeffs_0 = A0 \ ω₀.values[cell_ids]

    # R₁
    A1, rows_edges, cols_key = build_whitney_matrix_for_coarse_triangle(
        coarse_mesh, fine_mesh, coarse_tri; degree=2, k=k, nq=nq)
    edge_ids, edge_orients   = first.(rows_edges), last.(rows_edges)
    _, col_keep, A_ind       = independent_columns(A1; rtol=rtol)
    dω₀      = d(fine_mesh, ω₀)
    rhs      = dω₀.values[edge_ids] .* edge_orients
    coeffs_1 = A_ind \ rhs
    ψ        = combined_whitney_field(coarse_mesh, coarse_tri, coeffs_1, cols_key[col_keep], 2)

    # Test at vertices + centroid + edge midpoints
    cpts, _, _ = get_triangle(coarse_mesh, coarse_tri)
    verts      = [Float64[p[1], p[2]] for p in cpts]
    centroid   = (verts[1] .+ verts[2] .+ verts[3]) ./ 3
    mids       = [0.5 .* (verts[1] .+ verts[2]),
                  0.5 .* (verts[2] .+ verts[3]),
                  0.5 .* (verts[3] .+ verts[1])]
    test_pts   = vcat(verts, [centroid], mids)

    diffs = [maximum(abs.(grad_R0_at(p, coeffs_0, alphas_0, geom) .- ψ(p))) for p in test_pts]
    return maximum(diffs), (size_A0=size(A0),     cond_A0=cond(A0),
                            size_A1=size(A_ind),  cond_A1=cond(A_ind))
end

# (1) Polynomial ω̃ ∈ P_2  →  commutation must hit machine precision
coarse_tri = 11
ω_poly(x) = 1.0 + 2.0*x[1] - 3.0*x[2] + 0.5*x[1]*x[2] + 0.7*x[1]^2 - 0.4*x[2]^2
err_p, info_p = commutation_error_deg2(coarse_mesh, fine_mesh, coarse_tri, ω_poly)

println("Polynomial  ω̃ ∈ P_2          |d R₀ω₀ − R₁ d̄ω₀|∞  = ", err_p)
println("   A₀ $(info_p.size_A0)  cond ≈ ", round(info_p.cond_A0; sigdigits=3))
println("   A₁ $(info_p.size_A1)  cond ≈ ", round(info_p.cond_A1; sigdigits=3))

# (2) Non-polynomial smooth ω̃  →  does the LSQ diagram still commute?
ω_trig(x) = sin(2π*x[1])*cos(2π*x[2])
err_t, _ = commutation_error_deg2(coarse_mesh, fine_mesh, coarse_tri, ω_trig)
println("Non-poly    ω̃ = sin·cos      |d R₀ω₀ − R₁ d̄ω₀|∞  = ", err_t)

### Why the commutation closes exactly at degree 2

#### Local geometry on a coarse triangle

Subdividing each coarse triangle through the three primal-edge midpoints (the **medial subdivision**) gives exactly **6 small vertices** and **9 small edges**:

- 3 coarse vertices $V_1, V_2, V_3$ (the coarse cell centres);
- 3 coarse edge midpoints $M_{12}, M_{23}, M_{31}$;
- 6 boundary half-edges $+$ 3 medial edges $= 9$ small edges.

This count is purely combinatorial — it holds for *any* coarse triangle, regular or irregular.

#### Local cochain and reconstruction spaces

On one coarse triangle $T$ we have

| symbol | space | dim |
|---|---|---|
| $V_6 \;=\; C^0_p(T)$ | local 0-cochains (values at the 6 small vertices) | 6 |
| $V_9 \;=\; C^1_p(T)$ | local 1-cochains (line integrals along the 9 small edges) | 9 |
| $P_2\Lambda^0(T)$ | polynomials of degree $\leq 2$ in $\lambda_1,\lambda_2,\lambda_3$ | 6 |
| $P_2^{-}\Lambda^1(T)$ | $\{\lambda^{\beta} W_e : |\beta|=1\}$ modulo the single linear dependence | 8 |

and two reconstruction operators

$$
R_0 : V_6 \longrightarrow P_2\Lambda^0(T), \qquad R_1 : V_9 \longrightarrow P_2^{-}\Lambda^1(T) .
$$

$R_0$ is the **square** Lagrange system $A_0\,c = v$ with $A_0[i,\alpha] = \lambda^\alpha(p_i)$; unisolvence at the 6 Lagrange nodes makes it a **bijection**.
$R_1$ is the **rank-revealing LSQ** $A_1\,c = \omega$ with $A_1[e,(\beta,e')] = \int_e \lambda^\beta W_{e'}\cdot d\ell$; with rank 8 on 9 equations it is an **exact projection** onto $P_2^{-}\Lambda^1$.

#### The commutation diagram

$$
\begin{array}{ccc}
V_6 & \xrightarrow{\;\;\bar d\;\;} & V_9 \\[3pt]
{\scriptstyle R_0}\!\downarrow & & \downarrow{\scriptstyle R_1} \\[3pt]
P_2\Lambda^0(T) & \xrightarrow{\;\;d\;\;} & P_2^{-}\Lambda^1(T)
\end{array}
$$

**Claim.** $\;d\circ R_0 \;=\; R_1\circ \bar d\;$ as linear operators $V_6 \to P_2^{-}\Lambda^1(T)$.

#### Proof in three steps

Pick any 0-cochain $v\in V_6$.

**Step 1 — exact polynomial interpolation.**
By unisolvence of $P_2\Lambda^0$ at the 6 Lagrange nodes, there is a unique $p\in P_2\Lambda^0(T)$ with $p(p_i) = v_i$, $i=1,\dots,6$. By definition of $R_0$,
$$
R_0(v) \;=\; p, \qquad d\,R_0(v) \;=\; \nabla p .
$$

**Step 2 — FTC turns cochain differences into edge integrals of $\nabla p$.**
For every small edge $e$ with endpoints $c_1, c_2$ (both among the 6 vertices),
$$
(\bar d\,v)[e] \;=\; v_{c_2} - v_{c_1} \;=\; p(c_2) - p(c_1) \;=\; \int_e \nabla p \cdot d\ell ,
$$
the last equality being the **fundamental theorem of calculus**, *exact* because $p$ is a polynomial. Hence
$$
\bar d\,v \;=\; \Bigl(\, \textstyle\int_e \nabla p\cdot d\ell \,\Bigr)_{e\in\text{9 small edges}} \;\in\; V_9 .
$$

**Step 3 — $R_1$ is exact on $\nabla p$.**
Since $\nabla p \in P_1\Lambda^1 \subset P_2^{-}\Lambda^1$, and $R_1$ projects exactly onto $P_2^{-}\Lambda^1$,
$$
R_1(\bar d\,v) \;=\; R_1\!\Bigl(\bigl(\,\textstyle\int_e\nabla p\cdot d\ell\,\bigr)_e\Bigr) \;=\; \nabla p .
$$

Combining the three steps,
$$
R_1(\bar d\,v) \;=\; \nabla p \;=\; d\,R_0(v) \qquad \forall\,v\in V_6 . \qquad\blacksquare
$$

#### Why each ingredient is needed

| ingredient | role in the proof |
|---|---|
| $\dim V_6 \;=\; \dim P_2\Lambda^0 \;=\; 6$ | makes $R_0$ a **bijection** in Step 1 |
| $p\in P_2$ polynomial | makes FTC in Step 2 **exact**, not just a quadrature approximation |
| $\nabla\!\bigl(P_2\Lambda^0\bigr) \;\subset\; P_2^{-}\Lambda^1$ | the image of Step 2 lies in the range of $R_1$, so Step 3 is exact |
| 9 equations, rank 8 in $A_1$ | $R_1$ is a true projection onto $P_2^{-}\Lambda^1$, not just a fit |

#### Why mesh irregularity does not break it

Nowhere in the proof did we use the *shape* of the coarse triangle — only

1. the medial subdivision always yields 6 vertices and 9 edges (a combinatorial fact about subdividing a triangle by its three primal-edge midpoints), and
2. those 6 small vertices coincide with the standard P₂ Lagrange nodes (a geometric fact built into the fine mesh through `vcat(coarse_mesh.cells.position, coarse_mesh.edges.position)`).

Both facts hold on regular *and* irregular meshes, so the test in the previous code cell prints machine precision in either case.

### Check for convergence

In [ ]:
error_list = []
h_list = []
vector_field(x) = [sin(2*pi*x[1]) * cos(2*pi*x[2]), -cos(2*pi*x[1]) * sin(2*pi*x[2])]

for i = 1:5
    println("Mesh $i")
    coarse_mesh = VoronoiMesh("meshes/mesh$(i)_regular.nc")
    fine_mesh = VoronoiMesh(vcat(coarse_mesh.cells.position, coarse_mesh.edges.position),
                            coarse_mesh.x_period, coarse_mesh.y_period; max_iter=0)
    n_tris = coarse_mesh.vertices.n
    results = zeros(n_tris)
    # Discretize the field once on the fine mesh
    fine_v = discretize_vector_field_dual(fine_mesh, vector_field)

    # compute Whitney matrices for all coarse triangles
    data = build_whitney_matrices_for_mesh(coarse_mesh, fine_mesh)

    for coarse_tri in 1:n_tris
        A = data.A[coarse_tri]
        rows_edges = data.rows_edges[coarse_tri]
        cols_key = data.cols_key[coarse_tri]
        edge_ids = first.(rows_edges)
        edge_orients = last.(rows_edges)

        # Right-hand side and apply rank-revealing solve
        rhs = fine_v.values[edge_ids] .* edge_orients
        _, col_keep, A_ind = independent_columns(A; rtol=1e-6)

        # Solve with pseudoinverse of reduced matrix
        coeffs = A_ind \ rhs

        # Reconstruct the field on the coarse triangle
        approximation = combined_whitney_field(coarse_mesh, coarse_tri, coeffs, cols_key[col_keep], 2)
        p = coarse_mesh.vertices.position[coarse_tri]
        error = maximum(abs.(approximation(p) - vector_field(p)))
        results[coarse_tri] = error
    end
    push!(error_list, maximum(results))
    push!(h_list, maximum(coarse_mesh.edges.lengthDual))
end


### Observed convergence rate

The next two cells report the empirical order of the degree-2 Whitney reconstruction: `slope` is the last-pair log-log slope of `error_list` against `h_list`, and the final cell lists the per-mesh errors.

In [ ]:
slope = log(error_list[end]/error_list[end-1])/log(h_list[end]/h_list[end-1])

In [ ]:
error_list


## $R_0$ — reconstruct a $P_2\Lambda^0$ scalar from 6 fine-cell values

**Setup.** On one coarse triangle $T$ the medial subdivision exposes **6 fine cells**:
the 3 coarse vertices plus the 3 fine cells that sit at the **primal $\cap$ dual intersection** of each coarse edge (these are *not* edge midpoints in general).

**Target space.** $P_2\Lambda^0(T)$ has dimension 6, spanned by the monic monomials in barycentric coordinates:
$$
\bigl\{\lambda^\alpha : |\alpha|=2\bigr\} \;=\; \{\lambda_0^2,\ \lambda_1^2,\ \lambda_2^2,\ \lambda_0\lambda_1,\ \lambda_1\lambda_2,\ \lambda_2\lambda_0\}.
$$

**The system.** Define the $6\times 6$ matrix
$$
A_0[i,\alpha] \;:=\; \lambda^\alpha(p_i), \qquad p_i \in \text{6 fine cells}.
$$
For a discrete 0-cochain $\omega_0\in V_6$ with values $v_i = \omega_0[p_i]$, the system $A_0\,c = v$ is **square**, and it has a unique solution exactly when the 6 nodes are *unisolvent* for $P_2\Lambda^0$. Unisolvence holds generically; only degenerate node configurations (5 nodes on a quadric curve) fail. So in the generic case $R_0$ is a true **bijection** $V_6 \to P_2\Lambda^0(T)$, not a least-squares fit.

The non-midpoint geometry shows up only through $\operatorname{cond}(A_0)$ — the projector still exists, it just gets stiffer as the coarse triangle skews.

**Reconstructed field.**
$$
R_0(\omega_0)(p) \;=\; \sum_{|\alpha|=2} c_\alpha\,\lambda^\alpha(p), \qquad c = A_0^{-1}\,v.
$$

**Exactness.** For any $\tilde\omega^0\in P_2\Lambda^0$, $R_0(\tilde\omega^0|_{\text{cells}}) = \tilde\omega^0$ at machine precision (polynomial reproduction).

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
#   R₀ :  V_6  →  P_2 Λ⁰    on a single coarse triangle
# ─────────────────────────────────────────────────────────────────────────

"""
    R0_build(coarse_mesh, fine_mesh, coarse_tri; k=4)

Step-by-step:
  1.  Cache the coarse triangle geometry and the barycentric map.
  2.  Collect the 6 unique fine cells inside this coarse triangle
      (union of `fine_mesh.vertices.cells[fid]` over its 4 medial fine triangles).
  3.  Assemble the 6 × 6 monomial matrix  A₀[i, α] = λ^α(p_i)
      with α running over the 6 multi-indices of |α| = 2.

Returns `(A0, cell_ids, alphas, geom)` where `geom = (xA,yA,xB,yB,xC,yC,denom)`
encodes the coarse triangle and lets `R0_at` evaluate the polynomial later.
"""
function R0_build(coarse_mesh, fine_mesh, coarse_tri; k=4, small_tris=nothing)
    # ── 1. Coarse triangle geometry + barycentric map  (vertices wrapped to anchor) ──
    xp, yp     = coarse_mesh.x_period, coarse_mesh.y_period
    anchor     = coarse_mesh.vertices.position[coarse_tri]
    cpts, _, _ = get_triangle(coarse_mesh, coarse_tri)
    cp1 = closest(anchor, TensorsLite.Vec(x=cpts[1][1], y=cpts[1][2]), xp, yp)
    cp2 = closest(anchor, TensorsLite.Vec(x=cpts[2][1], y=cpts[2][2]), xp, yp)
    cp3 = closest(anchor, TensorsLite.Vec(x=cpts[3][1], y=cpts[3][2]), xp, yp)
    xA, yA = cp1.x, cp1.y
    xB, yB = cp2.x, cp2.y
    xC, yC = cp3.x, cp3.y
    denom  = (yB-yC)*xA + (yC-yA)*xB + (yA-yB)*xC
    bary(p) = (
        ((yB-yC)*p[1] + (xC-xB)*p[2] + xB*yC - xC*yB) / denom,
        ((yC-yA)*p[1] + (xA-xC)*p[2] + xC*yA - xA*yC) / denom,
        ((yA-yB)*p[1] + (xB-xA)*p[2] + xA*yB - xB*yA) / denom,
    )

    # ── 2. The 6 unique fine cells inside this coarse triangle ──────────
    #     `small_tris` may be precomputed once per mesh and passed in to avoid
    #     the O(n_coarse · n_fine) global scan inside `map_coarse_to_fine`.
    if small_tris === nothing
        small_tris, _, _ = map_coarse_to_fine(coarse_mesh, fine_mesh; k=k)
    end
    cell_ids = unique(reduce(vcat,
        [collect(fine_mesh.vertices.cells[fid]) for fid in small_tris[coarse_tri]]))

    # ── 3. 6×6 monomial matrix  A₀[i,α] = λ^α(p_i) ──────────────────────
    alphas = generate_multi_indices(2)            # 6 indices with |α| = 2
    A0     = zeros(length(cell_ids), length(alphas))
    for (i, fcid) in enumerate(cell_ids)
        p_w = closest(anchor, fine_mesh.cells.position[fcid], xp, yp)
        λ   = bary([p_w.x, p_w.y])
        for (j, α) in enumerate(alphas)
            A0[i, j] = λ[1]^α[1] * λ[2]^α[2] * λ[3]^α[3]
        end
    end
    return A0, cell_ids, alphas, (xA, yA, xB, yB, xC, yC, denom)
end

"""
    R0_at(p, coeffs_0, alphas, geom)

Evaluate the reconstructed scalar  R₀(ω₀)(p) = Σ_α  c_α · λ^α(p).
"""
function R0_at(p, coeffs_0, alphas, geom)
    xA,yA,xB,yB,xC,yC,denom = geom
    l0 = ((yB-yC)*p[1] + (xC-xB)*p[2] + xB*yC - xC*yB) / denom
    l1 = ((yC-yA)*p[1] + (xA-xC)*p[2] + xC*yA - xA*yC) / denom
    l2 = ((yA-yB)*p[1] + (xB-xA)*p[2] + xA*yB - xB*yA) / denom
    s  = 0.0
    for (c, α) in zip(coeffs_0, alphas)
        s += c * l0^α[1] * l1^α[2] * l2^α[3]
    end
    return s
end

# ── Smoke test on one coarse triangle ───────────────────────────────────
# (1) Exactness:  ω̃ ∈ P_2  ⇒  R₀ recovers it to machine precision.
# (2) Approximation:  smooth ω̃  ⇒  pointwise error O(h^3) as the mesh refines.
coarse_tri_test = 1
A0, cell_ids, alphas, geom0 = R0_build(coarse_mesh, fine_mesh, coarse_tri_test)

# (1) polynomial in P₂ — must hit machine precision
ω̃_poly(x) = 1.7 + 2.1*x[1] - 0.9*x[2] + 0.3*x[1]*x[2] + 0.6*x[1]^2 - 0.4*x[2]^2
v_poly    = [ω̃_poly([fine_mesh.cells.position[fid].x,
                     fine_mesh.cells.position[fid].y]) for fid in cell_ids]
c_poly    = A0 \ v_poly

# (2) smooth non-polynomial
ω̃_trig(x) = sin(2π*x[1])*cos(2π*x[2]) + 0.5*x[1] - 0.3*x[2]^2
v_trig    = [ω̃_trig([fine_mesh.cells.position[fid].x,
                     fine_mesh.cells.position[fid].y]) for fid in cell_ids]
c_trig    = A0 \ v_trig

cpts, _, _ = get_triangle(coarse_mesh, coarse_tri_test)
verts      = [Float64[p[1], p[2]] for p in cpts]
centroid   = (verts[1] .+ verts[2] .+ verts[3]) ./ 3
test_pts   = vcat(verts, [centroid])

err_poly = maximum(abs(R0_at(p, c_poly, alphas, geom0) - ω̃_poly(p)) for p in test_pts)
err_trig = maximum(abs(R0_at(p, c_trig, alphas, geom0) - ω̃_trig(p)) for p in test_pts)

println("R₀ smoke test  (coarse_tri = $coarse_tri_test)")
println("   A₀ size = ", size(A0), "   cond(A₀) ≈ ", round(cond(A0); sigdigits=4))
println("   |R₀ − ω̃|∞   on polynomial ω̃ ∈ P_2     = ", err_poly)
println("   |R₀ − ω̃|∞   on trig ω̃ = sin·cos + …  = ", err_trig)

## $R_2$ — reconstruct a $P_1\Lambda^2$ scalar from 4 fine-triangle integrals

**Setup.** The medial subdivision of one coarse triangle has **4 fine triangles** $T_1,\dots,T_4$ (three corner triangles + one central). A 2-cochain $\omega_2\in V_4$ lists their integrals $\xi_i = \int_{T_i}\omega_2$.

**Target space.** $P_1\Lambda^2(T)$ has dimension 3. Concretely we represent a polynomial 2-form by its scalar coefficient against $dx\wedge dy$, and use the **coarse-triangle barycentric coordinates** as the $P_1$ basis:
$$
P_1\Lambda^2(T) \;=\; \mathrm{span}\bigl\{\lambda_0,\ \lambda_1,\ \lambda_2\bigr\}\;\otimes\; dx\wedge dy ,
\qquad \dim = 3 .
$$

**The system.** The $4\times 3$ matrix
$$
A_2[i,j] \;:=\; \int_{T_i}\lambda_j\,dA
$$
is **overdetermined** (4 fine triangles, 3 basis functions), so $R_2$ is a true LSQ projection onto $P_1\Lambda^2(T)$.

**Closed form for the matrix entries.** Because $\lambda_j$ is *linear* in $(x,y)$, the centroid rule is exact:
$$
\int_{T_i}\lambda_j\,dA \;=\; |T_i|\cdot \bar\lambda_j(T_i),\qquad
\bar\lambda_j(T_i) \;=\; \tfrac{1}{3}\sum_{v\in\text{vertices}(T_i)}\lambda_j(v).
$$
No quadrature is needed to build $A_2$.

**Reconstructed field.**
$$
R_2(\omega_2)(p) \;=\; \sum_{j=0}^{2} c_j\,\lambda_j(p), \qquad
c = A_2^{+}\,\xi \;\in\; \mathbb R^3 .
$$

**Exactness.** For any $\tilde\omega^2\in P_1$, the four fine-triangle integrals lie in the range of $A_2$, the LSQ residual is zero, and $R_2$ recovers $\tilde\omega^2$ exactly.

**Discretizing a continuous 2-form.** To feed $R_2$ with the integrals of a smooth $\tilde\omega^2$, the cell below includes a 7-point Dunavant rule (exact for polynomials of degree $\leq 5$) so the discretization error is far smaller than the LSQ projection error.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
#   R₂ :  V_4  →  P_1 Λ²    on a single coarse triangle
# ─────────────────────────────────────────────────────────────────────────

"""
    R2_build(coarse_mesh, fine_mesh, coarse_tri; k=4)

Step-by-step:
  1.  Cache the coarse triangle geometry and the barycentric map.
  2.  Collect the 4 fine triangles inside the coarse triangle.
  3.  Assemble the 4 × 3 matrix  A₂[i, j] = ∫_{T_i} λ_j dA
      using the closed form  |T_i| · (avg of λ_j at the 3 vertices of T_i)
      (exact, because λ_j is linear).

Returns `(A2, fine_tri_ids, fine_tri_areas, geom)`.
"""
function R2_build(coarse_mesh, fine_mesh, coarse_tri; k=4, small_tris=nothing)
    # ── 1. Coarse triangle geometry + barycentric map  (vertices wrapped to anchor) ──
    xp, yp     = coarse_mesh.x_period, coarse_mesh.y_period
    anchor     = coarse_mesh.vertices.position[coarse_tri]
    cpts, _, _ = get_triangle(coarse_mesh, coarse_tri)
    cp1 = closest(anchor, TensorsLite.Vec(x=cpts[1][1], y=cpts[1][2]), xp, yp)
    cp2 = closest(anchor, TensorsLite.Vec(x=cpts[2][1], y=cpts[2][2]), xp, yp)
    cp3 = closest(anchor, TensorsLite.Vec(x=cpts[3][1], y=cpts[3][2]), xp, yp)
    xA, yA = cp1.x, cp1.y
    xB, yB = cp2.x, cp2.y
    xC, yC = cp3.x, cp3.y
    denom  = (yB-yC)*xA + (yC-yA)*xB + (yA-yB)*xC
    bary(p) = (
        ((yB-yC)*p[1] + (xC-xB)*p[2] + xB*yC - xC*yB) / denom,
        ((yC-yA)*p[1] + (xA-xC)*p[2] + xC*yA - xA*yC) / denom,
        ((yA-yB)*p[1] + (xB-xA)*p[2] + xA*yB - xB*yA) / denom,
    )

    # ── 2. The 4 fine triangles in this coarse triangle ──────────────────
    #     `small_tris` may be precomputed once per mesh and passed in to avoid
    #     the O(n_coarse · n_fine) global scan inside `map_coarse_to_fine`.
    if small_tris === nothing
        small_tris, _, _ = map_coarse_to_fine(coarse_mesh, fine_mesh; k=k)
    end
    fine_tri_ids     = small_tris[coarse_tri]

    # ── 3. 4 × 3 matrix  A₂[i,j] = |T_i| · avg(λ_j over vertices of T_i) ─
    A2             = zeros(length(fine_tri_ids), 3)
    fine_tri_areas = Float64[]
    for (i, fid) in enumerate(fine_tri_ids)
        pts, _, _ = get_triangle(fine_mesh, fid)
        v1 = closest(anchor, TensorsLite.Vec(x=pts[1][1], y=pts[1][2]), xp, yp)
        v2 = closest(anchor, TensorsLite.Vec(x=pts[2][1], y=pts[2][2]), xp, yp)
        v3 = closest(anchor, TensorsLite.Vec(x=pts[3][1], y=pts[3][2]), xp, yp)
        area = 0.5 * abs((v2.x-v1.x)*(v3.y-v1.y) - (v3.x-v1.x)*(v2.y-v1.y))
        push!(fine_tri_areas, area)
        λv1, λv2, λv3 = bary([v1.x, v1.y]), bary([v2.x, v2.y]), bary([v3.x, v3.y])
        for j in 1:3
            A2[i, j] = area * (λv1[j] + λv2[j] + λv3[j]) / 3
        end
    end
    return A2, fine_tri_ids, fine_tri_areas, (xA, yA, xB, yB, xC, yC, denom)
end

"""
    R2_at(p, coeffs_2, geom)

Evaluate the reconstructed 2-form scalar  R₂(ω₂)(p) = c₀ λ_0(p) + c₁ λ_1(p) + c₂ λ_2(p).
"""
function R2_at(p, coeffs_2, geom)
    xA,yA,xB,yB,xC,yC,denom = geom
    l0 = ((yB-yC)*p[1] + (xC-xB)*p[2] + xB*yC - xC*yB) / denom
    l1 = ((yC-yA)*p[1] + (xA-xC)*p[2] + xC*yA - xA*yC) / denom
    l2 = ((yA-yB)*p[1] + (xB-xA)*p[2] + xA*yB - xB*yA) / denom
    return coeffs_2[1]*l0 + coeffs_2[2]*l1 + coeffs_2[3]*l2
end

# ── 7-point Dunavant rule, exact for polynomials of degree ≤ 5.  Used only
#    to discretize a continuous 2-form  ω̃² ↦ {∫_{T_i} ω̃²}_i.
# Exact closed-form constants — weights sum to 1 to ULP precision because the
# √15 in (155+√15) and (155−√15) cancel exactly when added in floating point.
const _SQ15 = sqrt(15.0)
const _DUN_AS = (9 - 2*_SQ15) / 21   #  small barycentric value ≈ 0.0597
const _DUN_BS = (6 + _SQ15) / 21     #  large barycentric value ≈ 0.4701
const _DUN_AL = (9 + 2*_SQ15) / 21   #  large barycentric value ≈ 0.7974
const _DUN_BL = (6 - _SQ15) / 21     #  small barycentric value ≈ 0.1013
const _DUN_WS = (155 + _SQ15) / 1200 #  weight for the (small, large, large) orbit
const _DUN_WL = (155 - _SQ15) / 1200 #  weight for the (large, small, small) orbit
const _DUNAVANT7_BARY = (
    (1/3, 1/3, 1/3),
    (_DUN_AS, _DUN_BS, _DUN_BS),
    (_DUN_BS, _DUN_AS, _DUN_BS),
    (_DUN_BS, _DUN_BS, _DUN_AS),
    (_DUN_AL, _DUN_BL, _DUN_BL),
    (_DUN_BL, _DUN_AL, _DUN_BL),
    (_DUN_BL, _DUN_BL, _DUN_AL),
)
const _DUNAVANT7_WTS = (
    9/40,
    _DUN_WS, _DUN_WS, _DUN_WS,
    _DUN_WL, _DUN_WL, _DUN_WL,
)
@assert abs(sum(_DUNAVANT7_WTS) - 1) < 1e-15  # sanity check on weight normalization

function integrate_2form_over_triangle(f, v1, v2, v3)
    area = 0.5 * abs((v2.x-v1.x)*(v3.y-v1.y) - (v3.x-v1.x)*(v2.y-v1.y))
    s    = 0.0
    for (b, w) in zip(_DUNAVANT7_BARY, _DUNAVANT7_WTS)
        px = b[1]*v1.x + b[2]*v2.x + b[3]*v3.x
        py = b[1]*v1.y + b[2]*v2.y + b[3]*v3.y
        s += w * f([px, py])
    end
    return s * area
end

# ── Smoke test on one coarse triangle ───────────────────────────────────
# (1) Exactness:  ω̃² ∈ P_1  ⇒  R₂ recovers it to machine precision.
# (2) Approximation:  smooth ω̃²  ⇒  pointwise error O(h^2) as the mesh refines.
coarse_tri_test = 4
A2, fine_tri_ids, fine_areas, geom2 = R2_build(coarse_mesh, fine_mesh, coarse_tri_test)

xp, yp = coarse_mesh.x_period, coarse_mesh.y_period
anchor = coarse_mesh.vertices.position[coarse_tri_test]

# Helper: integrate ω̃² over a fine triangle (returns ∫_{T_i} ω̃² dA).
function _discretize_2form_on_patch(ω̃, fine_tri_ids)
    ξ = zeros(length(fine_tri_ids))
    for (i, fid) in enumerate(fine_tri_ids)
        pts, _, _ = get_triangle(fine_mesh, fid)
        v1 = closest(anchor, TensorsLite.Vec(x=pts[1][1], y=pts[1][2]), xp, yp)
        v2 = closest(anchor, TensorsLite.Vec(x=pts[2][1], y=pts[2][2]), xp, yp)
        v3 = closest(anchor, TensorsLite.Vec(x=pts[3][1], y=pts[3][2]), xp, yp)
        ξ[i] = integrate_2form_over_triangle(ω̃, v1, v2, v3)
    end
    return ξ
end

# (1) ω̃² ∈ P_1 — must hit machine precision
ω̃2_poly(x) = 0.7 - 1.2*x[1] + 2.3*x[2]
ξ_poly     = _discretize_2form_on_patch(ω̃2_poly, fine_tri_ids)
c2_poly    = A2 \ ξ_poly

# (2) Smooth non-polynomial ω̃²
ω̃2_trig(x) = sin(2π*x[1]) - 0.3*x[2] + 0.4
ξ_trig    = _discretize_2form_on_patch(ω̃2_trig, fine_tri_ids)
c2_trig   = A2 \ ξ_trig

cpts, _, _ = get_triangle(coarse_mesh, coarse_tri_test)
verts      = [Float64[p[1], p[2]] for p in cpts]
centroid   = (verts[1] .+ verts[2] .+ verts[3]) ./ 3
test_pts   = vcat(verts, [centroid])

err_poly = maximum(abs(R2_at(p, c2_poly, geom2) - ω̃2_poly(p))  for p in test_pts)
err_trig = maximum(abs(R2_at(p, c2_trig, geom2) - ω̃2_trig(p))  for p in test_pts)

println("R₂ smoke test  (coarse_tri = $coarse_tri_test)")
println("   A₂ size = ", size(A2), "   cond(A₂) ≈ ", round(cond(A2); sigdigits=4))
println("   |R₂ − ω̃²|∞  on linear ω̃² ∈ P_1     = ", err_poly)
println("   |R₂ − ω̃²|∞  on trig ω̃² = sin·… + …  = ", err_trig)

## Convergence sweep for $R_0$ and $R_2$

Mirrors the convergence test you already use for $R_1$:

1. Loop over `meshes/mesh$(i)_regular.nc` for `i = 1:5`.
2. For each mesh:
   * build `fine_mesh` from `vcat(coarse_mesh.cells.position, coarse_mesh.edges.position)` (one fine vertex at every coarse cell-center and every coarse primal-dual edge intersection),
   * **discretize the test field once** on the fine mesh:
     * $R_0$ — pointwise sample $\tilde\omega_0$ at every fine cell centre,
     * $R_2$ — exact (Dunavant-7) integral of $\tilde\omega_2$ over every fine triangle,
   * **for every coarse triangle**, build the local LSQ system (`R0_build` / `R2_build`), solve, evaluate the reconstruction at the coarse circumcentre, and record the absolute pointwise error,
   * the **mesh error** is the max over all coarse triangles.
3. Report the rate from the last two refinements: $\text{slope} = \log(e_5/e_4)\,/\,\log(h_5/h_4)$.

**Expected rates** (best approximation on a quasi-uniform triangle of size $h$):

| projector | target space               | rate           | slope |
|-----------|----------------------------|----------------|-------|
| $R_0$     | $P_2\Lambda^0$  (dim 6)    | $\mathcal O(h^3)$ | **3** |
| $R_1$     | $P_2^{-}\Lambda^1$ (dim 8) | $\mathcal O(h^2)$ | **2** *(matches your earlier result)* |
| $R_2$     | $P_1\Lambda^2$  (dim 3)    | $\mathcal O(h^2)$ | **2** |

$R_0$ is *higher* order than $R_1$ because $P_2\Lambda^0$ contains the full $P_2$, while $P_2^{-}\Lambda^1$ is the Koszul-trimmed space (it contains $P_1\Lambda^1$ exactly, capping the rate at $h^2$). $R_2$ targets $P_1$ (scalar), so its rate is $h^2$ as well.

In [ ]:
# ── Convergence sweep: R₀ : V_6 → P_2Λ⁰ ─────────────────────────────────
# Smooth, periodic test scalar (so the wrap on boundary triangles is harmless).
ω̃_0(x) = sin(2π*x[1]) * cos(2π*x[2])

R0_error_list = Float64[]
R0_h_list     = Float64[]

for i in 1:5
    println("Mesh $i")
    coarse_mesh = VoronoiMesh("meshes/mesh$(i)_regular.nc")
    fine_mesh   = VoronoiMesh(vcat(coarse_mesh.cells.position, coarse_mesh.edges.position),
                              coarse_mesh.x_period, coarse_mesh.y_period; max_iter=0)

    n_tris = coarse_mesh.vertices.n
    errors = zeros(n_tris)

    # Sample ω̃_0 once at every fine cell centre.
    fine_v = [ω̃_0([p.x, p.y]) for p in fine_mesh.cells.position]

    # ⚠ Precompute the coarse→fine triangle map ONCE per mesh (O(n_coarse · n_fine)).
    #   Without this, each call to `R0_build` re-scans every fine triangle, turning
    #   the per-mesh cost from O(n²) into O(n³).  This is the cell that used to be slow.
    small_tris, _, _ = map_coarse_to_fine(coarse_mesh, fine_mesh; k=4)

    for coarse_tri in 1:n_tris
        A0, cell_ids, alphas, geom0 = R0_build(coarse_mesh, fine_mesh, coarse_tri;
                                               small_tris=small_tris)
        coeffs0 = A0 \ fine_v[cell_ids]

        # The coarse circumcentre IS the anchor used inside R0_build,
        # so it is already in the anchor frame — no extra wrapping needed.
        anchor = coarse_mesh.vertices.position[coarse_tri]
        p_test = [anchor.x, anchor.y]
        errors[coarse_tri] = abs(R0_at(p_test, coeffs0, alphas, geom0) - ω̃_0(p_test))
    end

    push!(R0_error_list, maximum(errors))
    push!(R0_h_list,     maximum(coarse_mesh.edges.lengthDual))
end

R0_slope = log(R0_error_list[end] / R0_error_list[end-1]) /
           log(R0_h_list[end]     / R0_h_list[end-1])

println("\nR₀ convergence")
println("   h         = ", round.(R0_h_list;     sigdigits=4))
println("   |err|∞    = ", round.(R0_error_list; sigdigits=4))
println("   slope (last pair) = ", round(R0_slope; sigdigits=4), "   (expected ≈ 3)")

### $R_2$ convergence sweep

The mesh sweep for the 2-form reconstruction $R_2:V_4\to P_1\Lambda^2$ (4 fine-triangle integrals → degree-1 area form), using `R2_build`/`R2_at` on a periodic scalar. Expected slope $\approx 2$. (The cell just above runs the analogous $R_0$ sweep.)

In [ ]:
# ── Convergence sweep: R₂ : V_4 → P_1Λ² ─────────────────────────────────
# Periodic scalar coefficient ω̃_2 of the test 2-form  ω̃_2(x) · dx ∧ dy.
ω̃_2(x) = sin(2π*x[1]) * cos(2π*x[2])

R2_error_list = Float64[]
R2_h_list     = Float64[]

for i in 1:5
    println("Mesh $i")
    coarse_mesh = VoronoiMesh("meshes/mesh$(i)_regular.nc")
    fine_mesh   = VoronoiMesh(vcat(coarse_mesh.cells.position, coarse_mesh.edges.position),
                              coarse_mesh.x_period, coarse_mesh.y_period; max_iter=0)

    n_tris = coarse_mesh.vertices.n
    n_fine = fine_mesh.vertices.n
    errors = zeros(n_tris)

    # Exact (Dunavant-7) integral of ω̃_2 over every fine triangle.
    # `get_triangle` already wraps each fine triangle's vertices to its own
    # circumcentre, so no extra wrapping is needed here.
    fine_xi = zeros(n_fine)
    for ftri in 1:n_fine
        pts, _, _ = get_triangle(fine_mesh, ftri)
        v1 = TensorsLite.Vec(x=pts[1][1], y=pts[1][2])
        v2 = TensorsLite.Vec(x=pts[2][1], y=pts[2][2])
        v3 = TensorsLite.Vec(x=pts[3][1], y=pts[3][2])
        fine_xi[ftri] = integrate_2form_over_triangle(ω̃_2, v1, v2, v3)
    end

    # ⚠ Precompute the coarse→fine triangle map ONCE per mesh (O(n_coarse · n_fine)).
    #   Without this, each call to `R2_build` re-scans every fine triangle, turning
    #   the per-mesh cost from O(n²) into O(n³).  This is the cell that used to be slow.
    small_tris, _, _ = map_coarse_to_fine(coarse_mesh, fine_mesh; k=4)

    for coarse_tri in 1:n_tris
        A2, fine_tri_ids, _, geom2 = R2_build(coarse_mesh, fine_mesh, coarse_tri;
                                              small_tris=small_tris)
        coeffs2 = A2 \ fine_xi[fine_tri_ids]

        anchor = coarse_mesh.vertices.position[coarse_tri]
        p_test = [anchor.x, anchor.y]
        errors[coarse_tri] = abs(R2_at(p_test, coeffs2, geom2) - ω̃_2(p_test))
    end

    push!(R2_error_list, maximum(errors))
    push!(R2_h_list,     maximum(coarse_mesh.edges.lengthDual))
end

R2_slope = log(R2_error_list[end] / R2_error_list[end-1]) /
           log(R2_h_list[end]     / R2_h_list[end-1])

println("\nR₂ convergence")
println("   h         = ", round.(R2_h_list;     sigdigits=4))
println("   |err|∞    = ", round.(R2_error_list; sigdigits=4))
println("   slope (last pair) = ", round(R2_slope; sigdigits=4), "   (expected ≈ 2)")

## Commutation: does $d \circ I_h^{k} = I_h^{k+1} \circ \bar d_k$?

This section fixes the notation, states every operator with its domain and
codomain, derives the exact $1\to2$ defect formula step by step (so the role of
$A_2^{+}$ is explicit), and explains why the residual is exactly zero for
$\omega\in V^1$, nonzero-but-convergent for a smooth $\omega\notin V^1$, and why a
*non-periodic* test field can spuriously break the identity at the periodic
boundary.

---

### 1. Geometric setup (one coarse triangle)

Work on the periodic square $\Omega = \mathbb{R}^2/(L_x\mathbb{Z}\times L_y\mathbb{Z})$.
The **coarse** mesh is the Delaunay triangulation (the *primal* complex); each
coarse triangle $T$ has a circumcenter $c_T$ (a dual $0$-cell, stored in
`mesh.vertices.position`). Subdividing $T$ *medially* through the primal-dual
intersection points on its three edges produces a **fine patch** with

* $6$ fine cell centres = $3$ coarse vertices $+\;3$ edge-intersection points,
* $9$ fine primal edges,
* $4$ fine triangles.

**Periodic wrapping.** For an anchor $a\in\Omega$ define the *nearest-image map*
$$
\pi_a:\Omega\to\mathbb{R}^2,\qquad
\pi_a(x)=x-(m\,L_x,\;n\,L_y),\quad (m,n)\in\mathbb{Z}^2 \text{ chosen so } \pi_a(x) \text{ is the lift of } x \text{ closest to } a .
$$
In code this is `closest(a, x, xp, yp)`. The image $\{\pi_a(x):x\in\Omega\}$ is the
**tile** around $a$. Every geometric object attached to $T$ — its vertices, the
fine sub-simplices, and the polynomial basis of §2 — is built in the tile of
$c_T$, i.e. through $\pi_{c_T}$. Keep this anchor in mind: it is the heart of §6.3.

---

### 2. The two spaces, and the operators between them

For each $k$ there are **two different spaces** on the patch. They were previously
both written $V^k$, which is the source of much confusion; we now keep them apart.

| symbol | name | $k=0$ | $k=1$ | $k=2$ |
|:------:|------|:-----:|:-----:|:-----:|
| $\mathcal{C}_h^k$ | **fine cochain space** (the data you sample) | $6$ point values | $9$ edge integrals | $4$ triangle integrals |
| $V^k$ | **polynomial target space** (FEEC space you reconstruct into) | $P_2\Lambda^0,\ \dim 6$ | $P_2^{-}\Lambda^1,\ \dim 8$ | $P_1\Lambda^2,\ \dim 3$ |

The operators, with explicit domain $\to$ codomain:

**(a) de Rham / cochain sampling** $\;\mathcal{R}_h^k:\Lambda^k(T)\to\mathcal{C}_h^k$, taking a smooth $k$-form to its dofs on the fine patch:
$$
(\mathcal{R}_h^0\omega)_p=\omega(x_p)\ \ (6\text{ centres}),\quad
(\mathcal{R}_h^1\omega)_e=\!\int_{e}\!\omega\ \ (9\text{ edges}),\quad
(\mathcal{R}_h^2\omega)_t=\!\int_{t}\!\omega\ \ (4\text{ triangles}).
$$
Each fine cell is lifted to the tile of $c_T$ before integrating (this is what makes §3 exact).

**(b) reconstruction (LSQ)** $\;I_h^k:\mathcal{C}_h^k\to V^k$. Let $\{\phi_j^k\}_{j}$ be the basis of $V^k$ and define the **dof matrix**
$$
A_k[i,j]=\bigl(\mathcal{R}_h^k\,\phi_j^k\bigr)_i \quad\Longrightarrow\quad
I_h^k(\xi)=\sum_j c_j\,\phi_j^k,\qquad \mathbf{c}=A_k^{+}\,\xi .
$$
That is, $I_h^k$ **= "apply $A_k^{+}$, then expand in the basis"**; $A_k^{+}$ is the
inverse when $A_k$ is square and the least-squares pseudoinverse when it is tall. Sizes:
$$
A_0\in\mathbb{R}^{6\times 6}\ \text{(square)},\qquad
A_1\in\mathbb{R}^{9\times 8}\ \text{(tall)},\qquad
A_2\in\mathbb{R}^{4\times 3}\ \text{(tall)} .
$$
In code: $A_0$ = `R0_build`, $A_1$ = `build_whitney_matrix_for_coarse_triangle`, $A_2$ = `R2_build`.

**(c) discrete coboundary** $\;\bar d_k:\mathcal{C}_h^k\to\mathcal{C}_h^{k+1}$ — the signed incidence sum (transpose of the incidence matrix). Purely combinatorial; it does not see geometry. In code it is `d(mesh, ·)` restricted to the patch.

**(d) exterior derivative** $\;d:V^k\to\Lambda^{k+1}$ — the genuine continuous derivative of the reconstructed polynomial form (for $k=1$ this is the analytic `combined_whitney_curl`).

**(e) projector** $\;\Pi_h^k:=I_h^k\circ\mathcal{R}_h^k:\Lambda^k(T)\to V^k$.

---

### 3. Exact cochain identity (always true, combinatorial)

On fine cochains the discrete coboundary reproduces Stokes / FTC **exactly** for any smooth $\omega$:
$$
\bar d_k\circ\mathcal{R}_h^k \;=\; \mathcal{R}_h^{k+1}\circ d
\qquad\text{on } \mathcal{C}_h^k .
$$
$k=0$ is the fundamental theorem of calculus on each fine edge; $k=1$ is Stokes
on each fine triangle. **Caveat:** the boundary of a fine cell must be a closed
loop *in one tile*; if different fine edges of a coarse triangle were lifted to
different tiles the loop would not close and this identity would break (§6.3).

---

### 4. The commutation question

We ask whether the exact cochain identity **lifts through the reconstruction**:
$$
d\circ I_h^{k}\;\stackrel{?}{=}\;I_h^{k+1}\circ\bar d_k .
$$
The left side differentiates the reconstructed polynomial; the right side
differentiates the data first (combinatorially) and reconstructs afterwards.

---

### 5. Level $0\to1$ — exact for any smooth $\omega$

Because $A_0$ is **square**, $I_h^0$ is a genuine interpolant: $\omega_h:=\Pi_h^0\omega\in P_2\Lambda^0$ matches $\omega$ at all $6$ fine centres. FTC on every fine edge gives $\mathcal{R}_h^1(d\omega)=\mathcal{R}_h^1(d\omega_h)$, and since $d\omega_h\in V^1$ the level-$1$ LSQ recovers it without residual. Hence
$$
d\,I_h^0\,\mathcal{R}_h^0(\omega)\;=\;I_h^1\,\bar d_0\,\mathcal{R}_h^0(\omega)
\qquad\text{(machine precision).}
$$

---

### 6. Level $1\to2$ — the exact defect formula

#### 6.1 The level-1 residual ($A_1$ is tall)

Fix $\xi:=\mathcal{R}_h^1(\omega)\in\mathcal{C}_h^1$ ($9$ numbers). Since $\dim V^1=8$, $A_1$ is $9\times 8$ and the LSQ projects $\xi$ onto $\operatorname{col}(A_1)$. Split it:
$$
\xi \;=\; \underbrace{A_1\mathbf{c}}_{P_{\operatorname{col}(A_1)}\xi} \;+\; r_1,
\qquad \mathbf{c}=A_1^{+}\xi,
\qquad r_1:=\bigl(I-P_{\operatorname{col}(A_1)}\bigr)\xi \;\perp\;\operatorname{col}(A_1).
$$
$r_1$ is the part of the data **no field in $V^1$ can match**. If $\omega\in V^1$ then $\xi\in\operatorname{col}(A_1)$ and $r_1=0$.

#### 6.2 Derivation (this is where $A_2^{+}$ comes from)

Let $u:=I_h^1\xi\in V^1$ be the reconstructed field. By construction its own cochain is
$$
\mathcal{R}_h^1(u)=A_1\mathbf{c}=\xi-r_1 .
$$
Two **exact** facts:
* **Stokes on $u$** (§3): $\;\mathcal{R}_h^2(du)=\bar d_1\,\mathcal{R}_h^1(u)=\bar d_1(\xi-r_1)=\bar d_1\xi-\bar d_1 r_1.$
* **$du\in V^2$ is reconstructed exactly:** $du=\operatorname{curl}u\in P_1\Lambda^2=V^2$ and $A_2$ has full column rank, so $I_h^2\circ\mathcal{R}_h^2=\mathrm{id}$ on $V^2$. Hence
$$
du = I_h^2\mathcal{R}_h^2(du) = I_h^2(\bar d_1\xi-\bar d_1 r_1)= I_h^2\bar d_1\xi - I_h^2\bar d_1 r_1 .
$$
Since the left side is $du=d\,I_h^1\xi$, rearranging gives $d\,I_h^1\xi - I_h^2\bar d_1\xi = -\,I_h^2\bar d_1 r_1$. Writing $I_h^2(\eta)=\sum_j (A_2^{+}\eta)_j\psi_j^2$, the coefficient vector of the defect is
$$
\boxed{\;
d\,I_h^1\,\mathcal{R}_h^1(\omega)\;-\;I_h^2\,\bar d_1\,\mathcal{R}_h^1(\omega)
\;=\;-\,A_2^{+}\,\bar d_1\,r_1 .
\;}
$$

#### 6.3 Reading the formula, and why $A_2^{+}$ (not $A_1^{+}$)

Operator by operator:
* $r_1\in\mathcal{C}_h^1$ — the unrepresentable part of the level-1 data ($9$-vector $\perp\operatorname{col}A_1$);
* $\bar d_1 r_1\in\mathcal{C}_h^2$ — its **discrete curl**: the circulation of that leftover around each of the $4$ fine triangles;
* $A_2^{+}(\cdot)$ — the **level-2 reconstruction** $I_h^2$, turning that $4$-vector into coefficients of a polynomial $2$-form in $V^2$.

So the defect is literally *"reconstruct, into $V^2$, the discrete curl of the part of the data that $I_h^1$ could not represent."* The reason it is $A_2^{+}$ and **not** $A_1^{+}$: the level-1 pseudoinverse is *already inside* $r_1=(I-A_1A_1^{+})\xi$; the only operator still acting after the residual is formed is the level-2 reconstruction. $A_2$ is well-conditioned ($4\times3$, full rank), so it merely transports $\bar d_1 r_1$ into $V^2$ — it does not create the error, it only measures it. The defect is driven entirely by the level-1 residual $r_1$.

#### 6.4 Three regimes

* **$\omega\in V^1$ (e.g. $F=(-xy,x^2)$):** $r_1=0$ $\Rightarrow$ commutation **exact** ($\approx 10^{-13}$).
* **smooth $\omega\notin V^1$ (e.g. $\sin\cos$):** $r_1\neq 0$ is a genuine approximation error; the defect $-A_2^{+}\bar d_1 r_1$ is nonzero but $\to 0$ under refinement (slope $\ge 1$; see §7).
* **non-periodic $\omega\in V^1$ sampled by the global operators:** a *spurious* $r_1\neq0$ appears only on periodic-boundary triangles — see §6.5.

#### 6.5 The periodic-boundary subtlety (a sampling artifact, not a flaw)

Everything above assumes $\xi$ and $\bar d_1\xi$ are sampled in **the same tile** that
defines $V^1,V^2$, namely the tile of $c_T$: $\;\xi_e=\int_{\pi_{c_T}(e)}\omega.$
The *global* operators `discretize_vector_field_dual` and `d` instead lift each
fine edge $e$ to the tile of its **own** primal-dual point $x_e=$`mesh.edges.position[e]`,
i.e. $\int_{\pi_{x_e}(e)}\omega$.

* **Interior** coarse triangle: $\pi_{x_e}=\pi_{c_T}$ on all its edges — the two coincide.
* **Seam-crossing** coarse triangle: some fine edges land in a *neighbouring* tile.

Then:
* If $\omega$ is **periodic** (single-valued on the torus), $\int_{\pi_{x_e}(e)}\omega=\int_{\pi_{c_T}(e)}\omega$ — the mismatch is invisible and the global cochains are correct (this is the case for real fields and for the smooth-convergence test).
* If $\omega$ is a **non-periodic** test polynomial such as $F=(-xy,x^2)$, the two lifts disagree on seam edges, so $\xi\notin\operatorname{col}(A_1)$ **even though $F\in V^1$**. This manufactures a spurious $r_1\neq 0$, and the boxed formula returns a nonzero $-A_2^{+}\bar d_1 r_1$ — the $\approx 2$ residual previously seen on the $\sim 21$ boundary triangles (interior triangles were already at machine precision).

**Fix (used in the test cell).** Sample the local cochains in the $c_T$ tile and
build level-2 from them directly:
$$
\xi_e=\int_{\pi_{c_T}(e)}F\quad(\texttt{edge\_int\_local}),\qquad
(\bar d_1\xi)_t=\sum_{e\in\partial t}\pm\,\xi_e .
$$
Now $\xi,\bar d_1\xi$ live in one tile, Stokes closes, $r_1=0$ for $F\in V^1$, and the commutation is exact ($\approx 10^{-13}$) on **every** triangle.

> **Takeaway.** The DEC construction is correct and the commutation holds locally.
> The earlier $\approx 2$ defect was a *sampling artifact* of validating with a
> non-periodic field through tile-by-tile global operators — not a flaw in
> $I_h^k$, $\bar d_k$, or $d$.

---

### 7. Convergence of the projector $\Pi_h^k=I_h^k\circ\mathcal{R}_h^k$

Polynomial reproduction on $V^k$ together with uniform LSQ stability gives the Bramble-Hilbert bound
$$
\|\Pi_h^k\omega-\omega\|_{L^\infty(T)}
\;\le\;C\,\inf_{v\in V^k}\|\omega-v\|_{L^\infty(T)},
$$
hence the rates

| $k$ | $V^k$ | $L^\infty$ rate | observed slope |
|:---:|:----------------------|:---------------:|:--------------:|
| 0 | $P_2\Lambda^0$ | $h^3$ | **3** |
| 1 | $P_2^{-}\Lambda^1$ | $h^2$ | **2** |
| 2 | $P_1\Lambda^2$ | $h^2$ | **2** |

$P_2^{-}\Lambda^1$ contains all of $P_1\Lambda^1$ but only a $2$-dimensional Koszul
slice of degree-$2$ forms, which is why $I_h^1$ is one order lower than $I_h^0$.
For a smooth (periodic) $\omega\notin V^1$ the $1\to2$ commutation residual then
inherits the rate of $r_1$ and $\to 0$ at slope $\ge 1$ under refinement.

### Analytical curl of the reconstructed 1-form

Defines the **exact** (quadrature-free) curl of $I_h^1\xi=\sum c_{e,\alpha}\,\lambda^\alpha W_e$ through `combined_whitney_curl` / `whitney_curl_basis_at`, using
$\operatorname{curl}(\lambda^\alpha W_e)=\nabla\lambda^\alpha\times W_e+\lambda^\alpha\operatorname{curl}W_e$ with $\operatorname{curl}W_e=2\,d\lambda_a\wedge d\lambda_b$ (helper `grad_monomial`). This replaces the earlier numerical curl; **run it before** the commutation tests below.

In [ ]:
# ── Analytical curl of I_h¹ ω = Σ c_{e,α} λ^α W_e  (run this before commutation tests) ──
function grad_monomial(lambda, alpha, dl)
    gx, gy = 0.0, 0.0
    for i in 1:3
        if alpha[i] > 0
            factor = eval_monomial(lambda, alpha) / lambda[i]
            gx += alpha[i] * factor * dl[i][1]
            gy += alpha[i] * factor * dl[i][2]
        end
    end
    return gx, gy
end

function whitney_curl_basis_at(mesh, tri_idx, global_edge_idx, alpha, p, degree)
    _, edge_indices, edge_orientations = get_triangle(mesh, tri_idx)
    k = findfirst(==(global_edge_idx), edge_indices)
    k === nothing && error("edge $global_edge_idx not in coarse triangle $tri_idx")
    orient = edge_orientations[k]
    edge_pairs = ((1, 2), (2, 3), (3, 1))
    ia, ib = edge_pairs[k]

    center = mesh.vertices.position[tri_idx]
    xp, yp = mesh.x_period, mesh.y_period
    cpts, _, _ = get_triangle(mesh, tri_idx)
    p1 = closest(center, TensorsLite.Vec(x=cpts[1][1], y=cpts[1][2]), xp, yp)
    p2 = closest(center, TensorsLite.Vec(x=cpts[2][1], y=cpts[2][2]), xp, yp)
    p3 = closest(center, TensorsLite.Vec(x=cpts[3][1], y=cpts[3][2]), xp, yp)
    x_c = closest(center, TensorsLite.Vec(x=p[1], y=p[2]), xp, yp)
    xA, yA = p1.x, p1.y; xB, yB = p2.x, p2.y; xC, yC = p3.x, p3.y
    denom = (yB-yC)*xA + (yC-yA)*xB + (yA-yB)*xC

    l = [
        ((yB-yC)*x_c.x + (xC-xB)*x_c.y + xB*yC - xC*yB) / denom,
        ((yC-yA)*x_c.x + (xA-xC)*x_c.y + xC*yA - xA*yC) / denom,
        ((yA-yB)*x_c.x + (xB-xA)*x_c.y + xA*yB - xB*yA) / denom,
    ]
    dl = [
        [yB-yC, xC-xB] ./ denom,
        [yC-yA, xA-xC] ./ denom,
        [yA-yB, xB-xA] ./ denom,
    ]

    Wa, Wb = dl[ib], dl[ia]
    Wx = l[ia]*Wa[1] - l[ib]*Wb[1]
    Wy = l[ia]*Wa[2] - l[ib]*Wb[2]
    poly = eval_monomial(l, alpha)
    gx, gy = grad_monomial(l, alpha, dl)
    curl_W = 2.0 * (dl[ia][1]*dl[ib][2] - dl[ia][2]*dl[ib][1])
    return orient * (gx * Wy - gy * Wx + poly * curl_W)
end

function combined_whitney_curl(mesh, tri_idx, coeffs, cols_key, degree)
    return p -> sum(c * whitney_curl_basis_at(mesh, tri_idx, eid, alpha, p, degree)
                    for (c, (eid, alpha)) in zip(coeffs, cols_key))
end

### Sharp $1\to2$ commutation test on $F\in V^1$

Takes $F=(-xy,x^2)\in P_2^-\Lambda^1$ (Koszul-admissible, with $\operatorname{curl}F=3x\in V^2$) and checks $d\,I_h^1F = I_h^2\bar d_1F$ on every coarse triangle. The crucial detail: each triangle's local cochains are integrated in its **own circumcenter tile** via `edge_int_local`, so this non-periodic field is sampled consistently and the residual reaches $\sim10^{-13}$. The 2-cochain is built by discrete Stokes; reconstruction via `combined_whitney_curl` and `R2_build`/`R2_at`.

In [ ]:
# ── Sharp 1→2 commutation test: F ∈ V¹ ─────────────────────────────────────
# F = (-xy, x²) ∈ P₂⁻Λ¹  (Koszul check: xF₁+yF₂ = -x²y+x²y = 0 ✓),  curl F = 3x ∈ V².
#
# IMPORTANT (periodic boundary).  F is a *non-periodic* polynomial, so it takes
# different values in different periodic tiles.  The reconstruction I_h¹ lives in
# the coarse triangle's own tile (its vertices are wrapped to the circumcenter),
# so the discrete cochains must sample F in that SAME tile.  The global operators
# discretize_vector_field_dual / d wrap each fine edge to its own primal–dual
# point (mesh.edges.position); for the ~20% of coarse triangles that straddle a
# periodic seam this is a *different* tile, so F is sampled inconsistently there
# and the commutation appears to fail (residual ≈ 2).  Sampling the local
# cochains in the circumcenter tile removes this artifact, and the commutation is
# exact (≈ 1e-13) on every triangle, boundary ones included.

F_poly(x)          = [-x[1]*x[2], x[1]^2]
curl_F_analytic(x) = 3*x[1]

coarse_mesh = VoronoiMesh("meshes/mesh2_regular.nc")
fine_mesh   = VoronoiMesh(vcat(coarse_mesh.cells.position, coarse_mesh.edges.position),
                          coarse_mesh.x_period, coarse_mesh.y_period; max_iter=0)
n_tris = coarse_mesh.vertices.n
xp, yp = coarse_mesh.x_period, coarse_mesh.y_period

data = build_whitney_matrices_for_mesh(coarse_mesh, fine_mesh)
small_tris, _, _ = map_coarse_to_fine(coarse_mesh, fine_mesh; k=4)

# ∫_e F·dl over fine edge `feid`, endpoints wrapped to the coarse triangle's own
# periodic tile (its circumcenter) — the SAME tile the Whitney basis lives in.
edge_int_local(circ, feid) = begin
    v1, v2 = fine_mesh.edges.cells[feid]
    p1 = closest(circ, fine_mesh.cells.position[v1], xp, yp)
    p2 = closest(circ, fine_mesh.cells.position[v2], xp, yp)
    edge_integral(F_poly, [p1[1], p1[2]], [p2[1], p2[2]], xp, yp)
end

errs_rhs = zeros(n_tris)
errs_lhs = zeros(n_tris)
errs_com = zeros(n_tris)

for coarse_tri in 1:n_tris
    circ       = coarse_mesh.vertices.position[coarse_tri]
    A          = data.A[coarse_tri]
    rows_edges = data.rows_edges[coarse_tri]
    cols_key   = data.cols_key[coarse_tri]
    edge_ids   = first.(rows_edges)
    edge_orients = last.(rows_edges)

    # 1-cochain  R_h¹F  on this coarse triangle, sampled in its own tile
    rhs1 = [edge_int_local(circ, edge_ids[i]) * edge_orients[i] for i in eachindex(edge_ids)]

    _, col_keep, A_ind = independent_columns(A; rtol=1e-6)
    coeffs1 = A_ind \ rhs1
    curl_I1 = combined_whitney_curl(coarse_mesh, coarse_tri, coeffs1, cols_key[col_keep], 2)

    # 2-cochain  d̄₁ R_h¹F  = signed sum of the SAME 1-cochain around each fine
    # triangle (discrete Stokes), keeping both sides in one consistent tile.
    A2, fine_tri_ids, _, geom2 = R2_build(coarse_mesh, fine_mesh, coarse_tri; small_tris=small_tris)
    twocochain = zeros(length(fine_tri_ids))
    for (i, fid) in enumerate(fine_tri_ids)
        _, fedges, forients = get_triangle(fine_mesh, fid)
        twocochain[i] = sum(forients[k] * edge_int_local(circ, fedges[k]) for k in 1:3)
    end
    coeffs2 = A2 \ twocochain

    # centroid (get_triangle already returns vertices wrapped to `circ`)
    cpts, _, _ = get_triangle(coarse_mesh, coarse_tri)
    centroid = [(cpts[1][1]+cpts[2][1]+cpts[3][1])/3, (cpts[1][2]+cpts[2][2]+cpts[3][2])/3]

    g_I1   = curl_I1(centroid)
    g_I2   = R2_at(centroid, coeffs2, geom2)
    g_true = curl_F_analytic(centroid)

    errs_rhs[coarse_tri] = abs(g_I1 - g_true)
    errs_lhs[coarse_tri] = abs(g_I2 - g_true)
    errs_com[coarse_tri] = abs(g_I1 - g_I2)
end

println("Sharp 1→2 test: F = (-xy, x²) ∈ V¹   (curl F = 3x ∈ V²)")
println("   max |d I_h¹ F − curl F |        = ", maximum(errs_rhs))
println("   max |I_h² d̄₁ F − curl F |       = ", maximum(errs_lhs))
println("   max |d I_h¹ F − I_h² d̄₁ F|      = ", maximum(errs_com),
        "    ← commutation residual (expect ≈ 0)")

### Convergence of the $1\to2$ commutation residual

Measures $\lVert d\,I_h^1\xi - I_h^2\bar d_1\xi\rVert_\infty$ over a mesh sweep for a **periodic** field $F\notin V^1$ (so no boundary-tile artifact). Reuses `build_whitney_matrices_for_mesh`, `combined_whitney_curl`, and `R2_build`/`R2_at`. Expected slope $\approx 1$: the defect equals $A_2^{+}\bar d_1 r_1$ with $r_1=O(h)$ the level-1 LSQ residual.

In [ ]:
# ── Convergence of the 1→2 commutation residual on a smooth F ∉ V¹ ──────
# F = sin(2πx)cos(2πy) is periodic.  Both A and fine_v use edge_pos-relative
# wrapping ⇒ consistent short-path integrals for every periodic boundary edge.
# Failure formula:  ‖d I_h¹ ω − I_h² d̄₁ ω‖∞  =  ‖A₂⁺ d̄₁ r₁‖∞
#   where r₁ = (I − P_{col(A₁)}) R_h¹(ω) is the level-1 LSQ residual.
vec_field(x) = [ sin(2π*x[1]) * cos(2π*x[2]),
                -cos(2π*x[1]) * sin(2π*x[2])]

comm_err_list = Float64[]
comm_h_list   = Float64[]

for i in 1:4
    println("Mesh $i")
    coarse_mesh = VoronoiMesh("meshes/mesh$(i)_regular.nc")
    fine_mesh   = VoronoiMesh(vcat(coarse_mesh.cells.position, coarse_mesh.edges.position),
                              coarse_mesh.x_period, coarse_mesh.y_period; max_iter=0)
    n_tris  = coarse_mesh.vertices.n
    errors  = zeros(n_tris)

    fine_v  = discretize_vector_field_dual(fine_mesh, vec_field)
    fine_dω = d(fine_mesh, fine_v)

    data = build_whitney_matrices_for_mesh(coarse_mesh, fine_mesh)
    small_tris, _, _ = map_coarse_to_fine(coarse_mesh, fine_mesh; k=4)

    for coarse_tri in 1:n_tris
        A          = data.A[coarse_tri]
        rows_edges = data.rows_edges[coarse_tri]
        cols_key   = data.cols_key[coarse_tri]
        edge_ids   = first.(rows_edges)
        edge_orients = last.(rows_edges)

        rhs1 = fine_v.values[edge_ids] .* edge_orients

        _, col_keep, A_ind = independent_columns(A; rtol=1e-6)
        coeffs1 = A_ind \ rhs1
        curl_I1 = combined_whitney_curl(coarse_mesh, coarse_tri, coeffs1,
                                        cols_key[col_keep], 2)

        A2, fine_tri_ids, _, geom2 = R2_build(coarse_mesh, fine_mesh, coarse_tri;
                                              small_tris=small_tris)
        coeffs2 = A2 \ fine_dω.values[fine_tri_ids]

        cpts, _, _ = get_triangle(coarse_mesh, coarse_tri)
        centroid = [(cpts[1][1]+cpts[2][1]+cpts[3][1])/3,
                    (cpts[1][2]+cpts[2][2]+cpts[3][2])/3]

        errors[coarse_tri] = abs(curl_I1(centroid) - R2_at(centroid, coeffs2, geom2))
    end

    push!(comm_err_list, maximum(errors))
    push!(comm_h_list,   maximum(coarse_mesh.edges.lengthDual))
end

comm_slope = log(comm_err_list[end] / comm_err_list[end-1]) /
             log(comm_h_list[end]   / comm_h_list[end-1])

println("\n1→2 commutation residual convergence  (smooth F ∉ V¹)")
println("   h       = ", round.(comm_h_list;   sigdigits=4))
println("   |err|∞  = ", round.(comm_err_list; sigdigits=4))
println("   slope (last pair) = ", round(comm_slope; sigdigits=4),
        "   (expect ≥ 1)")

## Higher-order Whitney Hodge star $\star_1:\ \mathcal{C}^1_{\text{primal}}\to\mathcal{C}^1_{\text{dual}}$

Same construction as the lowest-order `whitney_hodge_star_operator` in
`juliaDEC.jl`, but the per-triangle field is the **higher-order lifting
reconstruction** $I_h^1$ (degree-2 trimmed Whitney space $P_2^{-}\Lambda^1$),
obtained by least squares — exactly the $A_1^{+}$ from the reconstruction work
above.

### The idea

In 2D the Hodge star on a $1$-form is a pointwise $90^\circ$ rotation: a field
$F=(F_x,F_y)$ becomes
$$
\star F = (-F_y,\;F_x).
$$
A **primal** 1-cochain stores circulations $\int_e F\cdot d\ell$ along primal
(Delaunay) edges; its Hodge star stores the values on the **dual** (Voronoi)
edges,
$$
(\star_1\alpha)_{e^\ast}=\int_{e^\ast}\star F\cdot d\ell .
$$
Each dual edge $e^\ast$ runs from circumcenter $c_{T_1}$ through
$x_e=$`edges.position[e]` to $c_{T_2}$, so it splits into one segment inside
$T_1$ and one inside $T_2$:
$$
(\star_1\alpha)_{e^\ast}
=\int_{c_{T_1}\to x_e}\!\!\star F_{T_1}\cdot d\ell
\;+\;\int_{x_e\to c_{T_2}}\!\!\star F_{T_2}\cdot d\ell ,
\qquad F_T = I_h^1(\alpha|_T).
$$

### How least squares enters

On a triangle $T$ the reconstruction is $F_T=\sum_k c_k\,\phi_k$ with
$\mathbf c = A_1^{+}\,\xi$ ($\xi$ = the fine stencil edge values, $A_1^{+}$ the
pseudoinverse). Writing $b_k=\int (\star\phi_k)\cdot d\ell$ over the dual
half-segment, the contribution is linear in $\xi$:
$$
\int \star F_T\cdot d\ell = b^{\top}\mathbf c = b^{\top}A_1^{+}\xi
= \bigl(A_1^{+\top}b\bigr)^{\top}\xi ,
$$
so the row of the Hodge-star matrix on that triangle is
$$
\boxed{\;w = A_1^{+\top}\,b\;}
$$
scattered to the triangle's global stencil edges (with the edge orientations
folded in). This is the only change from the canonical version: the canonical
basis is square (3 edges, exact), here $A_1$ is $9\times 8$ and $A_1^{+}$ is the
least-squares solve.

### Exactness

If the underlying field lies in $V^1=P_2^{-}\Lambda^1$ then $F_T=F$ exactly, so
$\star F_T=\star F$ and each dual-edge integral is reproduced to machine
precision. The validation cell below confirms this on the interior dual edges
(boundary edges carry the same non-periodic-field tile artifact discussed in the
commutation section; for a genuinely periodic field they too are exact up to the
$O(h^2)$ reconstruction error).

In [ ]:
# ── Higher-order Whitney Hodge star: fine primal 1-cochain → fine dual 1-cochain ──
# ⋆ on 1-forms in 2D: vector (a,b) ↦ (-b, a)   (90° rotation)
rot90(v) = (-v[2], v[1])

function ho_hodge_star_matrix(coarse_mesh, fine_mesh; degree=2, nq=5, rtol=1e-6)
    small_tris, small_edges, small_orients = map_coarse_to_fine(coarse_mesh, fine_mesh)
    xp, yp = coarse_mesh.x_period, coarse_mesh.y_period
    n_tri  = coarse_mesh.vertices.n
    pts, wts = gaussian_quadrature_points_and_weights(nq)

    # fine triangle (= fine vertex/circumcenter) -> its coarse triangle
    fine_tri_to_coarse = zeros(Int, fine_mesh.vertices.n)
    for t in 1:n_tri, ft in small_tris[t]
        fine_tri_to_coarse[Int(ft)] = t
    end

    # Cache the LSQ reconstruction data once per coarse triangle.
    cache = Vector{Any}(undef, n_tri)
    for t in 1:n_tri
        A, rows_edges, cols_key = build_whitney_matrix_for_coarse_triangle(
            coarse_mesh, fine_mesh, t, small_edges, small_orients; degree=degree, nq=nq)
        _, col_keep, A_ind = independent_columns(A; rtol=rtol)
        cache[t] = (Pt = pinv(A_ind)', cols_key = cols_key, col_keep = col_keep,
                    edge_ids = first.(rows_edges), edge_orients = last.(rows_edges))
    end

    # Weights on triangle t's 9 stencil edges for ∫_{p1→p2} (⋆ I_h¹ξ)·dl.
    seg_weights(t, p1, p2) = begin
        c   = cache[t]
        seg = (p2[1]-p1[1], p2[2]-p1[2])
        b   = zeros(length(c.col_keep))
        for (xi, wi) in zip(pts, wts)
            tt = (xi + 1)/2
            p  = TensorsLite.Vec(x = p1[1] + tt*seg[1], y = p1[2] + tt*seg[2])
            bd, _ = compute_higher_order_whitney_basis(coarse_mesh, t, p, degree)
            for (kk, kcol) in enumerate(c.col_keep)
                (eid, alpha) = c.cols_key[kcol]
                phi = bd[eid][alpha]
                b[kk] += wi * (-phi[2]*seg[1] + phi[1]*seg[2])   # (⋆φ)·seg
            end
        end
        b ./= 2                                  # [-1,1] → [0,1] Jacobian
        return (c.Pt * b) .* c.edge_orients, c.edge_ids   # row w = A₁⁺ᵀ b, orientation folded
    end

    I = Int[]; J = Int[]; V = Float64[]
    for fe in 1:fine_mesh.edges.n
        v1, v2 = fine_mesh.edges.vertices[fe]             # the two adjacent fine triangles
        xfe = fine_mesh.edges.position[fe]                # primal–dual intersection
        p1 = closest(xfe, fine_mesh.vertices.position[v1], xp, yp)
        p2 = closest(xfe, fine_mesh.vertices.position[v2], xp, yp)
        t1 = fine_tri_to_coarse[Int(v1)]; t2 = fine_tri_to_coarse[Int(v2)]   # split across coarse triangles
        w1, id1 = seg_weights(t1, [p1[1],p1[2]], [xfe[1],xfe[2]])
        w2, id2 = seg_weights(t2, [xfe[1],xfe[2]], [p2[1],p2[2]])
        for (w, id) in ((w1,id1), (w2,id2)), k in 1:length(id)
            push!(I, fe); push!(J, id[k]); push!(V, w[k])
        end
    end
    return sparse(I, J, V, fine_mesh.edges.n, fine_mesh.edges.n)
end

# ── validation: ⋆F for F = (-xy, x²) ∈ V¹  ⇒  exact on interior dual edges ──
let
    coarse_mesh = VoronoiMesh("meshes/mesh2_regular.nc")
    fine_mesh   = VoronoiMesh(vcat(coarse_mesh.cells.position, coarse_mesh.edges.position),
                              coarse_mesh.x_period, coarse_mesh.y_period; max_iter=0)
    xp, yp = coarse_mesh.x_period, coarse_mesh.y_period
    
    #F_poly(x) = [-1, 2]
    #starF(x)  = [-2, -1]
    
    F_poly(x) = [-x[1] - x[2], x[1]]
    starF(x)  = [-x[1], -x[1] - x[2]]            # analytic ⋆F as a vector field

    S    = ho_hodge_star_matrix(coarse_mesh, fine_mesh)
    xi   = discretize_vector_field_dual(fine_mesh, F_poly).values
    dual = S * xi

    errs = zeros(fine_mesh.edges.n)
    for fe in 1:fine_mesh.edges.n
        v1, v2 = fine_mesh.edges.vertices[fe]
        xfe = fine_mesh.edges.position[fe]
        p1 = closest(xfe, fine_mesh.vertices.position[v1], xp, yp)
        p2 = closest(xfe, fine_mesh.vertices.position[v2], xp, yp)
        ana = edge_integral(starF, [p1[1],p1[2]], [xfe[1],xfe[2]], xp, yp) +
              edge_integral(starF, [xfe[1],xfe[2]], [p2[1],p2[2]], xp, yp)
        errs[fe] = abs(dual[fe] - ana)
    end
    println("HO Hodge star  ⋆₁ : size ", size(S))
    println("error list:", errs)
end

### Convergence on a periodic field

For a field **not** in $V^1$ the per-triangle reconstruction $F_T=I_h^1(\alpha|_T)$ is only approximate, so $\star_h\alpha$ approximates the exact dual cochain $\mathcal R_h^{\text{dual}}(\star F)$ rather than reproducing it. Using a genuinely **periodic** $F$ removes the boundary-tile artifact (every fine dual edge, interior and boundary, is now consistent), so we can read a clean rate:

* `density slopes` $\to 2$ — error per unit dual length, i.e. the $O(h^2)$ accuracy of the $I_h^1$ reconstruction on $P_2^-\Lambda^1$;
* `abs slopes` $\to 3$ — the raw dual-edge integral picks up one extra power $O(h)$ from the shrinking edge length.

In [ ]:
# ── periodic-field convergence of the higher-order Hodge star ──────────────
# periodic, divergence-free field; its Hodge star is ⋆F = (-F2, F1)
F(x)     = [ sin(2π*x[1])*cos(2π*x[2]), -cos(2π*x[1])*sin(2π*x[2]) ]
starF(x) = [  cos(2π*x[1])*sin(2π*x[2]),  sin(2π*x[1])*cos(2π*x[2]) ]

hs = Float64[]; e_abs = Float64[]; e_den = Float64[]
for i in 1:5
    cm = VoronoiMesh("meshes/mesh$(i)_regular.nc")
    fm = VoronoiMesh(vcat(cm.cells.position, cm.edges.position), cm.x_period, cm.y_period; max_iter=0)
    xp, yp = cm.x_period, cm.y_period
    S  = ho_hodge_star_matrix(cm, fm)               # fine primal edge → fine dual edge
    xi = discretize_vector_field_dual(fm, F).values  # primal 1-cochain of F
    dual = S * xi                                    # discrete dual 1-cochain  ⋆ₕα

    err = zeros(fm.edges.n); den = zeros(fm.edges.n)
    for fe in 1:fm.edges.n
        v1, v2 = fm.edges.vertices[fe]; xfe = fm.edges.position[fe]
        p1 = closest(xfe, fm.vertices.position[v1], xp, yp)
        p2 = closest(xfe, fm.vertices.position[v2], xp, yp)
        # exact dual-edge integral of ⋆F, split at the primal-dual crossing xfe
        ana = edge_integral(starF, [p1[1],p1[2]], [xfe[1],xfe[2]], xp, yp) +
              edge_integral(starF, [xfe[1],xfe[2]], [p2[1],p2[2]], xp, yp)
        L = sqrt((p2[1]-p1[1])^2 + (p2[2]-p1[2])^2)
        err[fe] = abs(dual[fe] - ana)
        den[fe] = err[fe] / max(L, 1e-300)           # error per unit dual length
    end
    push!(hs, sum(cm.edges.length)/cm.edges.n)        # mean coarse primal edge length
    push!(e_abs, maximum(err)); push!(e_den, maximum(den))
    println("mesh $i:  h=", round(hs[end], digits=5),
            "   max|⋆ₕα-⋆α|=", e_abs[end],
            "   max(|⋆ₕα-⋆α|/Ldual)=", e_den[end])
end

conv_slope(e) = [log(e[k]/e[k-1])/log(hs[k]/hs[k-1]) for k in 2:length(e)]
println("\nh              = ", round.(hs, digits=5))
println("abs slopes     = ", round.(conv_slope(e_abs), digits=3), "  (≈ 3: O(h²) reconstruction × O(h) edge length)")
println("density slopes = ", round.(conv_slope(e_den), digits=3), "  (≈ 2: I_h¹ reconstruction order on P₂⁻Λ¹)")

In [ ]:
# ── Spectrum of the HO Hodge star S ─────────────────────────────────────────
# S is nonsymmetric (lift–apply–discretise), so eigenvalues may be complex.

cm = VoronoiMesh("meshes/mesh2_regular.nc")
fm = VoronoiMesh(vcat(cm.cells.position, cm.edges.position),
                 cm.x_period, cm.y_period; max_iter=0)
S  = Matrix(ho_hodge_star_matrix(cm, fm))
λ  = eigvals(S)

tol = 1e-12
n_cplx = count(z -> abs(imag(z)) > tol, λ)
println("n                  = ", size(S, 1))
println("||S - S'|| / ||S|| = ", opnorm(S - S') / opnorm(S))
println("# complex λ        = ", n_cplx, " / ", length(λ))
println("max |Im λ|         = ", maximum(abs.(imag.(λ))))
println("min / max Re λ     = ", extrema(real.(λ)))
println("all real?          = ", n_cplx == 0)

# compare: diagonal DEC ★₁ is symmetric ⇒ spectrum real
λd = eigvals(Matrix(star(fm, 1, true)))
println("\ndiagonal ★₁: max|Im λ| = ", maximum(abs.(imag.(λd))),
        "  (should be ~0)")


### Discrete Laplacian from the higher-order Hodge star

On the **fine** mesh (where `ho_hodge_star_matrix` acts), the Laplace–de Rham operator on primal 0-forms is

\[
\Delta_h \;=\; \star_2\, d\, \star_1\, d .
\]

Compare two choices of \(\star_1\):

* **diagonal DEC:** \(\operatorname{diag}(\ell^\ast/\ell)\) on fine edges;
* **higher-order Whitney:** \(S\) from `ho_hodge_star_matrix`.

\(\star_2\) and \(d\) are the standard DEC operators on the fine mesh. Periodic test field:

\[
u=\cos(4\pi x/L_x)\cos(4\pi y/L_y),\qquad
\Delta u=-16\pi^2\bigl(L_x^{-2}+L_y^{-2}\bigr)\,u.
\]


In [ ]:
# ── Discrete Laplacian: HO Whitney ★₁ vs diagonal DEC ★₁ ───────────────────
# Δ = ★₂ d ★₁ d   on the fine mesh (same complex; only ★₁ changes)

laplace_matrix(mesh, star1) =
    star(mesh, 2, false) * d(mesh, 1, false) * star1 * d(mesh, 0, true)

function laplacian_errors(cm, fm, S_ho)
    xp, yp = cm.x_period, cm.y_period
    L_ho   = laplace_matrix(fm, S_ho)
    L_diag = laplace_matrix(fm, star(fm, 1, true))

    u_fun(x)  = cos(4π * x[1]/xp) * cos(4π * x[2]/yp)
    Δu_fun(x) = -16*(π^2) * u_fun(x) * (xp^(-2) + yp^(-2))

    u     = [u_fun(p)  for p in fm.cells.position]
    Δtrue = [Δu_fun(p) for p in fm.cells.position]
    e_ho   = L_ho   * u .- Δtrue
    e_diag = L_diag * u .- Δtrue
    a = fm.cells.area
    l2(e) = sqrt(sum(a .* abs2.(e)))
    return (
        h = sum(cm.edges.length)/cm.edges.n,
        linf_ho = maximum(abs.(e_ho))/maximum(abs.(Δtrue)),   linf_diag = maximum(abs.(e_diag))/maximum(abs.(Δtrue)),
        l2_ho   = l2(e_ho)/l2(Δtrue),              l2_diag   = l2(e_diag)/l2(Δtrue),
        ker_ho  = maximum(abs.(L_ho * ones(fm.cells.n))),
        ker_diag = maximum(abs.(L_diag * ones(fm.cells.n))),
    )
end

hs = Float64[]; e∞_ho = Float64[]; e∞_d = Float64[]
e2_ho = Float64[]; e2_d = Float64[]

println(rpad("mesh",6), rpad("h",10),
        rpad("L∞ HO",14), rpad("L∞ diag",14),
        rpad("L² HO",14), rpad("L² diag",14))
for i in 1:4
    cm = VoronoiMesh("meshes/mesh$(i)_regular.nc")
    fm = VoronoiMesh(vcat(cm.cells.position, cm.edges.position),
                     cm.x_period, cm.y_period; max_iter=0)
    S  = ho_hodge_star_matrix(cm, fm)
    println("Is the second order hodge star symmetric? Answer: $(issymmetric(S))")
    r  = laplacian_errors(cm, fm, S)
    push!(hs, r.h); push!(e∞_ho, r.linf_ho); push!(e∞_d, r.linf_diag)
    push!(e2_ho, r.l2_ho); push!(e2_d, r.l2_diag)
    println(rpad("$i",6), rpad(string(round(r.h; digits=5)),10),
            rpad(string(round(r.linf_ho; sigdigits=8)),14),
            rpad(string(round(r.linf_diag; sigdigits=8)),14),
            rpad(string(round(r.l2_ho; sigdigits=8)),14),
            rpad(string(round(r.l2_diag; sigdigits=8)),14),
            "  ||L1||∞: HO=$(round(r.ker_ho; sigdigits=8)) diag=$(round(r.ker_diag; sigdigits=3))")
end

slopee(e) = [log(e[k]/e[k-1])/log(hs[k]/hs[k-1]) for k in 2:length(e)]
println("\nL∞ slopes  HO   = ", round.(slopee(e∞_ho); digits=8))
println("L∞ slopes  diag = ", round.(slopee(e∞_d); digits=8))
println("L² slopes  HO   = ", round.(slopee(e2_ho); digits=8))
println("L² slopes  diag = ", round.(slopee(e2_d); digits=8))


### Whitney consistency mass $M_{\mathrm{cons}}=(A^{+})^{\top} G A^{+}$

Pulls the \(L^2\) product on the Whitney space back to fine 1-cochains:

$$
(\xi,\eta)_{M}
\;=\;
\sum_T\bigl(I_h^T\xi,\,I_h^T\eta\bigr)_{L^2(T)}.
$$

Implementation: `whitney_consistency_mass.jl`
- `whitney_gram_matrix` — $G_{jk}=\int_T\phi_j\cdot\phi_k$ (Dunavant-7)
- `whitney_consistency_mass` — assembles $M_{\mathrm{cons}}$
- `whitney_mass_with_stabilisation` — optional $M_{\mathrm{cons}}+\alpha S_{\mathrm{stab}}$


In [ ]:
include("whitney_consistency_mass.jl")

cm = VoronoiMesh("meshes/mesh2.nc")
fm = VoronoiMesh(vcat(cm.cells.position, cm.edges.position),cm.x_period, cm.y_period; max_iter=0)
xp, yp = cm.x_period, cm.y_period

M = whitney_consistency_mass(cm, fm)

# SPD / SPSD check
λ = eigvals(Symmetric(Matrix(M)))
println("size(M) = ", size(M), "  nnz = ", nnz(M))
println("λmin(M) = ", minimum(λ), "   λmax(M) = ", maximum(λ))
println("# strictly negative = ", count(<( -1e-12), λ))

# Energy consistency: ξᵀ M ξ ≈ ∫ |F|²
function domain_L2(F)
    s = 0.0
    for t in 1:cm.vertices.n
        points, _, _ = get_triangle(cm, t)
        anchor = cm.vertices.position[t]
        p1 = closest(anchor, TensorsLite.Vec(x=points[1][1], y=points[1][2]), xp, yp)
        p2 = closest(anchor, TensorsLite.Vec(x=points[2][1], y=points[2][2]), xp, yp)
        p3 = closest(anchor, TensorsLite.Vec(x=points[3][1], y=points[3][2]), xp, yp)
        area = 0.5 * abs((p2.x-p1.x)*(p3.y-p1.y) - (p3.x-p1.x)*(p2.y-p1.y))
        for (bary, w) in zip(_DUNAVANT7_BARY, _DUNAVANT7_WTS)
            x = bary[1]*p1.x + bary[2]*p2.x + bary[3]*p3.x
            y = bary[1]*p1.y + bary[2]*p2.y + bary[3]*p3.y
            v = F([x, y])
            s += w * area * (v[1]^2 + v[2]^2)
        end
    end
    return s
end

println("\nEnergy consistency ξᵀMξ vs ∫|F|²:")
for (tag, F) in (
    ("constant (1,-1/2)",  x -> [1.0, -0.5]),
    ("linear (-x, x+y)",   x -> [-x[1], x[1] + x[2]]),
)
    xi = discretize_vector_field_dual(fm, F).values
    ed, ex = dot(xi, M * xi), domain_L2(F)
    println(rpad(tag, 22), "  rel err = ", abs(ed - ex) / max(ex, 1e-300))
end

# SPSD Laplacian
L = d(fm, 0, true)' * M * d(fm, 0, true)
μ = eigvals(Symmetric(Matrix(L)))
println("\nL = dᵀ M d :  λmin = ", minimum(μ),
        "   ||L 1||∞ = ", maximum(abs.(L * ones(fm.cells.n))))


In [ ]:
# ── Global energy = sum of per-triangle contributions (local charts) ───────
F(x) = [-x[1], x[1] + x[2]]   # linear, not periodic
degree = 2

_, small_edges, small_orients = map_coarse_to_fine(cm, fm)
xp, yp = cm.x_period, cm.y_period

e_rec_tot = 0.0
e_ex_tot  = 0.0

for t in 1:cm.vertices.n
    A, rows_edges, cols_key = build_whitney_matrix_for_coarse_triangle(
        cm, fm, t, small_edges, small_orients; degree=degree, nq=5)
    _, col_keep, A_ind = independent_columns(A; rtol=1e-6)
    G = whitney_gram_matrix(cm, t, cols_key; degree=degree)[col_keep, col_keep]

    # DoFs in THIS triangle's chart
    anchor = cm.vertices.position[t]
    ξor = zeros(length(rows_edges))
    for (i, (feid, forient)) in enumerate(rows_edges)
        v1, v2 = fm.edges.cells[feid]
        p1 = closest(anchor, fm.cells.position[v1], xp, yp)
        p2 = closest(anchor, fm.cells.position[v2], xp, yp)
        ξor[i] = forient * edge_integral(F, [p1[1],p1[2]], [p2[1],p2[2]], xp, yp)
    end

    c = pinv(A_ind) * ξor
    e_rec_tot += dot(c, G * c)

    # ∫_T |F|²
    points, _, _ = get_triangle(cm, t)
    p1 = closest(anchor, TensorsLite.Vec(x=points[1][1], y=points[1][2]), xp, yp)
    p2 = closest(anchor, TensorsLite.Vec(x=points[2][1], y=points[2][2]), xp, yp)
    p3 = closest(anchor, TensorsLite.Vec(x=points[3][1], y=points[3][2]), xp, yp)
    area = 0.5 * abs((p2.x-p1.x)*(p3.y-p1.y) - (p3.x-p1.x)*(p2.y-p1.y))
    for (bary, w) in zip(_DUNAVANT7_BARY, _DUNAVANT7_WTS)
        x = bary[1]*p1.x + bary[2]*p2.x + bary[3]*p3.x
        y = bary[1]*p1.y + bary[2]*p2.y + bary[3]*p3.y
        v = F([x, y])
        e_ex_tot += w * area * (v[1]^2 + v[2]^2)
    end
end

println("∑_T ∫|I_h F|² = ", e_rec_tot)
println("∑_T ∫|F|²     = ", e_ex_tot)
println("rel err       = ", abs(e_rec_tot - e_ex_tot) / e_ex_tot)